In [ ]:
"""
=============================================================================
PHASE 1 — DATA PREPARATION  (Google Drive + Checkpoint Version)
ECG Arrhythmia Classification for PYNQ-Z2 FPGA Deployment
=============================================================================

KEY CHANGES vs previous version:
  ✓ Everything saved to Google Drive — survives session disconnects
  ✓ Checkpoint system — resumes from last completed step automatically
  ✓ NO GPU needed — run on CPU runtime, save all GPU hours for Phase 2
  ✓ SMOTE n_jobs removed — compatible with all imblearn versions

HOW TO USE:
  1. Set runtime to CPU (Runtime → Change runtime type → None)
  2. Run all cells top to bottom
  3. If disconnected, reconnect and run again — it will skip already done steps
  4. After full completion, your Drive will have all .npy files ready for Phase 2

GOOGLE DRIVE FOLDER STRUCTURE CREATED:
  MyDrive/
  └── ECG_FPGA_Project/
      ├── phase1_outputs/         ← figures
      ├── X_train.npy
      ├── y_train.npy
      ├── X_val.npy
      ├── y_val.npy
      ├── X_test.npy
      ├── y_test.npy
      └── checkpoints/
          ├── step_filter_done.flag
          ├── step_smote_done.flag
          ├── X_filtered.npy      ← intermediate (auto-deleted after SMOTE)
          └── y_raw.npy           ← intermediate
=============================================================================
"""

# Switch runtime to CPU FIRST (Runtime → Change runtime type → None)
# Then run this cell.

!pip install -q imbalanced-learn
print("Libraries ready.")



from google.colab import drive
drive.mount('/content/drive')

import os

# ── All output goes here — change the folder name if you like ──────────────
DRIVE_ROOT   = '/content/drive/MyDrive/ECG_FPGA_Project'
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, 'checkpoints')
FIG_DIR        = os.path.join(DRIVE_ROOT, 'phase1_outputs')

for d in [DRIVE_ROOT, CHECKPOINT_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Drive mounted. Project folder: {DRIVE_ROOT}")
print("\nFolder structure:")
print(f"  {DRIVE_ROOT}/")
print(f"  ├── phase1_outputs/   ← figures")
print(f"  ├── checkpoints/      ← intermediate saves (resume support)")
print(f"  └── *.npy             ← final prepared arrays")



import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # non-interactive backend — safe for Colab
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from scipy.signal   import butter, filtfilt
from sklearn.model_selection import train_test_split
from sklearn.utils  import shuffle
from imblearn.over_sampling import SMOTE
from collections    import Counter
import time

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Dataset settings ───────────────────────────────────────────────────────
# Where are your CSV files?
# Option A: uploaded to /content/ via Files panel → upload
# Option B: stored in Drive (change path below)
CSV_DIR = '/content/drive/MyDrive/ECG_FPGA_Project/'                               # ← change if needed
# If your CSVs are in Drive already, use:
# CSV_DIR = '/content/drive/MyDrive/ECG_FPGA_Project/'

TRAIN_CSV = os.path.join(CSV_DIR, 'mitbih_train.csv')
TEST_CSV  = os.path.join(CSV_DIR, 'mitbih_test.csv')

# ── Signal settings ────────────────────────────────────────────────────────
N_SAMPLES    = 186            # time-steps per beat
LABEL_COL    = 187            # label column in the CSV
FS           = 125.0          # MIT-BIH sampling frequency (Hz)
LOWCUT       = 0.5            # band-pass lower cutoff (Hz)
HIGHCUT      = 40.0           # band-pass upper cutoff (Hz)
FILTER_ORDER = 4

# ── Balancing ──────────────────────────────────────────────────────────────
SMOTE_TARGET = 20000          # samples per class after balancing

# ── Split ──────────────────────────────────────────────────────────────────
VAL_SPLIT = 0.20              # 20% of balanced train → validation

# ── Class labels ───────────────────────────────────────────────────────────
CLASS_NAMES  = {0:'N — Normal', 1:'S — Supraventricular',
                2:'V — Ventricular', 3:'F — Fusion', 4:'Q — Unknown'}
CLASS_SHORT  = ['N','S','V','F','Q']
CLASS_COLORS = ['#4472C4','#ED7D31','#A9D18E','#E74C3C','#7030A0']

print("Configuration ready.")
print(f"  CSV directory  : {CSV_DIR}")
print(f"  Drive root     : {DRIVE_ROOT}")
print(f"  SMOTE target   : {SMOTE_TARGET:,} per class")
print(f"  Val split      : {int(VAL_SPLIT*100)}%")



def checkpoint_exists(name):
    """Return True if this step was already completed in a previous session."""
    flag = os.path.join(CHECKPOINT_DIR, f'{name}.flag')
    return os.path.exists(flag)

def save_checkpoint(name):
    """Mark a step as completed."""
    flag = os.path.join(CHECKPOINT_DIR, f'{name}.flag')
    with open(flag, 'w') as f:
        f.write(f'completed at {time.strftime("%Y-%m-%d %H:%M:%S")}')
    print(f"  ✓ Checkpoint saved: {name}")

def npy_path(filename):
    """Return full Drive path for a .npy file."""
    return os.path.join(DRIVE_ROOT, filename)

def ckpt_npy_path(filename):
    """Return full path for a checkpoint .npy file."""
    return os.path.join(CHECKPOINT_DIR, filename)

def save_fig(fig, filename):
    """Save a matplotlib figure to the Drive figures folder."""
    path = os.path.join(FIG_DIR, filename)
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved figure → {filename}")

print("Checkpoint helpers ready.")



print("\n" + "="*60)
print("STEP 1 — Loading MIT-BIH CSV files")
print("="*60)

train_raw = pd.read_csv(TRAIN_CSV, header=None)
test_raw  = pd.read_csv(TEST_CSV,  header=None)

train_raw[LABEL_COL] = train_raw[LABEL_COL].astype(int)
test_raw[LABEL_COL]  = test_raw[LABEL_COL].astype(int)

print(f"  Training CSV : {train_raw.shape}")
print(f"  Testing  CSV : {test_raw.shape}")

# Separate signals and labels
X_raw  = train_raw.iloc[:, :N_SAMPLES].values.astype(np.float32)
y_raw  = train_raw[LABEL_COL].values.astype(np.int64)
X_test = test_raw.iloc[:,  :N_SAMPLES].values.astype(np.float32)
y_test = test_raw[LABEL_COL].values.astype(np.int64)

# Show raw distribution
raw_counts = Counter(y_raw)
print(f"\n  Raw class distribution:")
for cls in range(5):
    cnt = raw_counts[cls]
    bar = '█' * (cnt // 2000)
    print(f"    Class {cls} ({CLASS_SHORT[cls]}): {cnt:>6,}  {bar}")

# ── Plot raw distribution ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(CLASS_SHORT, [raw_counts[i] for i in range(5)],
       color=CLASS_COLORS, edgecolor='white', linewidth=0.8)
ax.set_title('Class distribution — BEFORE balancing', fontweight='bold')
ax.set_xlabel('AAMI Class')
ax.set_ylabel('Number of beats')
for i in range(5):
    ax.text(i, raw_counts[i]+400, f'{raw_counts[i]:,}',
            ha='center', va='bottom', fontsize=9)
fig.tight_layout()
save_fig(fig, '01_raw_distribution.png')

print("\n  Step 1 complete.")


#           CHECKPOINT: skipped if already done in a previous session

print("\n" + "="*60)
print("STEP 2 — Band-pass filter (0.5–40 Hz)")
print("="*60)

FILTER_DONE = checkpoint_exists('step_filter')

if FILTER_DONE:
    # ── RESUME: load previously filtered arrays from Drive ─────────────────
    print("  ► Checkpoint found — loading filtered arrays from Drive...")
    X_train_filt = np.load(ckpt_npy_path('X_filtered_train.npy'))
    X_test_filt  = np.load(ckpt_npy_path('X_filtered_test.npy'))
    y_raw        = np.load(ckpt_npy_path('y_raw.npy'))
    print(f"  X_train_filt : {X_train_filt.shape}")
    print(f"  X_test_filt  : {X_test_filt.shape}")
    print("  Skipping filter step — already done.")

else:
    # ── RUN: apply filter for the first time ────────────────────────────────
    def bandpass_filter(signal_1d):
        """Zero-phase Butterworth band-pass filter for one beat."""
        nyq  = 0.5 * FS
        low  = LOWCUT  / nyq
        high = HIGHCUT / nyq
        b, a = butter(FILTER_ORDER, [low, high], btype='band')
        return filtfilt(b, a, signal_1d).astype(np.float32)

    print(f"  Filtering {X_raw.shape[0]:,} training beats ...")
    t0 = time.time()
    X_train_filt = np.array([bandpass_filter(x) for x in X_raw])
    print(f"  Training filter done in {time.time()-t0:.1f}s")

    print(f"  Filtering {X_test.shape[0]:,} test beats ...")
    t0 = time.time()
    X_test_filt  = np.array([bandpass_filter(x) for x in X_test])
    print(f"  Test filter done in {time.time()-t0:.1f}s")

    # Save intermediate filtered arrays to Drive (checkpoint)
    print("  Saving filtered arrays to Drive ...")
    np.save(ckpt_npy_path('X_filtered_train.npy'), X_train_filt)
    np.save(ckpt_npy_path('X_filtered_test.npy'),  X_test_filt)
    np.save(ckpt_npy_path('y_raw.npy'),             y_raw)
    save_checkpoint('step_filter')

    # ── Visualise filter effect on one sample per class ─────────────────────
    t_ms = np.arange(N_SAMPLES) / FS * 1000
    fig, axes = plt.subplots(5, 2, figsize=(13, 13))
    fig.suptitle('Band-pass filter effect (0.5–40 Hz) — one beat per class',
                 fontsize=11, fontweight='bold')
    for cls in range(5):
        idx = np.where(y_raw == cls)[0][0]
        axes[cls,0].plot(t_ms, X_raw[idx],        color='#4472C4', lw=0.8)
        axes[cls,0].set_title(f'{CLASS_SHORT[cls]} — Raw',      fontsize=9)
        axes[cls,1].plot(t_ms, X_train_filt[idx], color='#ED7D31', lw=0.8)
        axes[cls,1].set_title(f'{CLASS_SHORT[cls]} — Filtered', fontsize=9)
        for ax in axes[cls]:
            ax.set_ylim(-1.3, 1.3)
            ax.set_ylabel('Amp.')
    for ax in axes[-1]:
        ax.set_xlabel('Time (ms)')
    fig.tight_layout()
    save_fig(fig, '02_filter_effect.png')

print("  Step 2 complete.")


#           CHECKPOINT: skipped if already done in a previous session
#           This is the slowest step (~10–20 min on CPU)

print("\n" + "="*60)
print("STEP 3 — SMOTE balancing")
print("="*60)

SMOTE_DONE = checkpoint_exists('step_smote')

if SMOTE_DONE:
    # ── RESUME: load previously balanced arrays from Drive ─────────────────
    print("  ► Checkpoint found — loading balanced arrays from Drive...")
    X_balanced = np.load(ckpt_npy_path('X_balanced.npy'))
    y_balanced = np.load(ckpt_npy_path('y_balanced.npy'))
    print(f"  X_balanced : {X_balanced.shape}")
    print(f"  y_balanced : {y_balanced.shape}")
    print("  Skipping SMOTE step — already done.")

else:
    # ── RUN: apply SMOTE for the first time ─────────────────────────────────
    print(f"  Target: {SMOTE_TARGET:,} samples per class")
    print("  Running SMOTE (this takes ~10-20 min on CPU) ...")
    print("  You can leave this running — it will checkpoint when done.\n")

    # Build sampling strategy: only upsample classes below target
    # (don't downsample class 0 which already has 72,471 — we handle that after)
    current_counts = Counter(y_raw)
    sampling_strategy = {
        cls: SMOTE_TARGET
        for cls in range(5)
        if current_counts[cls] < SMOTE_TARGET
    }
    print(f"  Classes to upsample: {sampling_strategy}")

    t0 = time.time()
    smote = SMOTE(
        sampling_strategy=sampling_strategy,
        k_neighbors=5,
        random_state=SEED
        # n_jobs removed — compatible with all imblearn versions
    )
    X_smoted, y_smoted = smote.fit_resample(X_train_filt, y_raw)
    elapsed = time.time() - t0
    print(f"\n  SMOTE completed in {elapsed/60:.1f} minutes.")

    # SMOTE only upsamples minority classes.
    # Class 0 (Normal) still has 72,471 samples — we need to downsample it.
    # Randomly sample SMOTE_TARGET from class 0 to keep balance perfect.
    print(f"\n  Downsampling class 0 (Normal) from "
          f"{Counter(y_smoted)[0]:,} → {SMOTE_TARGET:,} ...")

    idx_class0 = np.where(y_smoted == 0)[0]
    rng = np.random.default_rng(SEED)
    keep_idx_0 = rng.choice(idx_class0, SMOTE_TARGET, replace=False)

    # Indices for all other classes (already at SMOTE_TARGET)
    idx_others = np.where(y_smoted != 0)[0]

    # Combine and shuffle
    all_idx    = np.concatenate([keep_idx_0, idx_others])
    X_balanced = X_smoted[all_idx]
    y_balanced = y_smoted[all_idx]
    X_balanced, y_balanced = shuffle(X_balanced, y_balanced, random_state=SEED)

    # Verify balance
    bal_counts = Counter(y_balanced)
    print(f"\n  Balanced class distribution:")
    for cls in range(5):
        cnt = bal_counts[cls]
        bar = '█' * (cnt // 1000)
        print(f"    Class {cls} ({CLASS_SHORT[cls]}): {cnt:>6,}  {bar}")

    # Save balanced arrays to Drive (checkpoint)
    print("\n  Saving balanced arrays to Drive ...")
    np.save(ckpt_npy_path('X_balanced.npy'), X_balanced)
    np.save(ckpt_npy_path('y_balanced.npy'), y_balanced)
    save_checkpoint('step_smote')

    # ── Plot balanced distribution ───────────────────────────────────────────
    bal_vals = [Counter(y_balanced)[i] for i in range(5)]
    fig, ax  = plt.subplots(figsize=(8, 4))
    ax.bar(CLASS_SHORT, bal_vals, color=CLASS_COLORS, edgecolor='white', lw=0.8)
    ax.set_title('Class distribution — AFTER SMOTE balancing', fontweight='bold')
    ax.set_xlabel('AAMI Class')
    ax.set_ylabel('Number of beats')
    ax.set_ylim(0, SMOTE_TARGET * 1.2)
    for i, cnt in enumerate(bal_vals):
        ax.text(i, cnt+100, f'{cnt:,}', ha='center', va='bottom', fontsize=9)
    fig.tight_layout()
    save_fig(fig, '03_balanced_distribution.png')

print("  Step 3 complete.")



print("\n" + "="*60)
print("STEP 4 — Train/Validation split + reshape for 1D-CNN")
print("="*60)

# ── Split balanced data into train and validation ──────────────────────────
# IMPORTANT: Test set (X_test_filt, y_test) is NEVER touched here.
# It stays separate until final evaluation in Phase 6.
X_train, X_val, y_train, y_val = train_test_split(
    X_balanced, y_balanced,
    test_size=VAL_SPLIT,
    random_state=SEED,
    stratify=y_balanced          # ensures equal class ratio in both splits
)

print(f"  Training   : {X_train.shape[0]:>7,} samples")
print(f"  Validation : {X_val.shape[0]:>7,} samples")
print(f"  Test (held): {X_test_filt.shape[0]:>7,} samples  ← DO NOT USE UNTIL PHASE 6")

# ── Verify class proportions in each split ─────────────────────────────────
print(f"\n  {'Class':<7} {'Train':>10} {'Val':>10} {'Test (raw)':>12}")
print("  " + "-"*43)
for cls in range(5):
    tr = np.sum(y_train == cls)
    vl = np.sum(y_val   == cls)
    te = np.sum(y_test  == cls)
    print(f"  {CLASS_SHORT[cls]:<7} {tr:>10,} {vl:>10,} {te:>12,}")

# ── Reshape: (N, 186) → (N, 186, 1)  ──────────────────────────────────────
X_train      = X_train.reshape(-1, N_SAMPLES, 1).astype(np.float32)
X_val        = X_val.reshape(-1, N_SAMPLES, 1).astype(np.float32)
X_test_final = X_test_filt.reshape(-1, N_SAMPLES, 1).astype(np.float32)

print(f"\n  After reshape:")
print(f"    X_train      : {X_train.shape}   dtype: {X_train.dtype}")
print(f"    X_val        : {X_val.shape}   dtype: {X_val.dtype}")
print(f"    X_test_final : {X_test_final.shape}  dtype: {X_test_final.dtype}")

# ── Signal range check ─────────────────────────────────────────────────────
print(f"\n  Signal value range:")
print(f"    X_train  min={X_train.min():.3f}  max={X_train.max():.3f}  mean={X_train.mean():.4f}")
print(f"    X_test   min={X_test_final.min():.3f}  max={X_test_final.max():.3f}  mean={X_test_final.mean():.4f}")

print("\n  Step 4 complete.")



print("\n" + "="*60)
print("STEP 5 — Saving final .npy arrays to Google Drive")
print("="*60)

save_map = {
    'X_train.npy' : X_train,
    'y_train.npy' : y_train,
    'X_val.npy'   : X_val,
    'y_val.npy'   : y_val,
    'X_test.npy'  : X_test_final,
    'y_test.npy'  : y_test,
}

for fname, arr in save_map.items():
    path   = npy_path(fname)
    np.save(path, arr)
    size   = os.path.getsize(path) / 1024 / 1024
    print(f"  ✓ {fname:<18} → {size:5.1f} MB   shape {arr.shape}")

print(f"\n  All files saved to: {DRIVE_ROOT}")



print("\n" + "="*60)
print("STEP 6 — Final visualisation")
print("="*60)

t_ms = np.arange(N_SAMPLES) / FS * 1000

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle('One beat per class — balanced training set (filtered + SMOTE)',
             fontsize=11, fontweight='bold')
for cls in range(5):
    idx  = np.where(y_train == cls)[0][0]
    beat = X_train[idx, :, 0]
    axes[cls].plot(t_ms, beat, color=CLASS_COLORS[cls], lw=1.0)
    axes[cls].set_title(CLASS_NAMES[cls], fontsize=8, fontweight='bold')
    axes[cls].set_xlabel('Time (ms)', fontsize=8)
    axes[cls].set_ylim(-1.3, 1.3)
    axes[cls].axhline(0, color='gray', lw=0.4, ls='--')
    axes[cls].grid(True, alpha=0.3, lw=0.4)
axes[0].set_ylabel('Amplitude (norm.)')
fig.tight_layout()
save_fig(fig, '04_final_sample_beats.png')

print("  Step 6 complete.")


#            to save Drive space (the .flag files stay so resume still works)

print("\n" + "="*60)
print("OPTIONAL — Cleaning up intermediate checkpoint arrays")
print("="*60)

intermediate_files = [
    'X_filtered_train.npy',
    'X_filtered_test.npy',
    'y_raw.npy',
    'X_balanced.npy',
    'y_balanced.npy',
]

total_freed = 0
for fname in intermediate_files:
    path = ckpt_npy_path(fname)
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024 / 1024
        os.remove(path)
        total_freed += size
        print(f"  Deleted: {fname}  ({size:.1f} MB)")
    else:
        print(f"  Not found (already deleted): {fname}")

print(f"\n  Total space freed: {total_freed:.1f} MB")
print("  Note: .flag files kept so resume still works if you rerun.")



print("\n" + "="*60)
print("PHASE 1 COMPLETE — SUMMARY")
print("="*60)

print(f"""
  Dataset     : MIT-BIH Arrhythmia (PhysioNet / Kaggle CSV)
  Classes     : 5 (AAMI EC57 — N, S, V, F, Q)
  Filter      : Butterworth band-pass {LOWCUT}–{HIGHCUT} Hz (order {FILTER_ORDER})
  Balancing   : SMOTE + class-0 downsample → {SMOTE_TARGET:,} per class
  Val split   : {int(VAL_SPLIT*100)}% from training data only (no test leakage)

  Final shapes saved to Google Drive:
    X_train : {X_train.shape}   y_train : {y_train.shape}
    X_val   : {X_val.shape}   y_val   : {y_val.shape}
    X_test  : {X_test_final.shape}  y_test  : {y_test.shape}

  Drive folder: {DRIVE_ROOT}

  ─────────────────────────────────────────────────────
  NEXT STEP — Phase 2: Model Training
  Change runtime to GPU (T4) and run Phase 2.
  Load your data at the top of Phase 2 with:

    from google.colab import drive
    drive.mount('/content/drive')
    import numpy as np
    ROOT    = '/content/drive/MyDrive/ECG_FPGA_Project/'
    X_train = np.load(ROOT + 'X_train.npy')
    y_train = np.load(ROOT + 'y_train.npy')
    X_val   = np.load(ROOT + 'X_val.npy')
    y_val   = np.load(ROOT + 'y_val.npy')
    X_test  = np.load(ROOT + 'X_test.npy')
    y_test  = np.load(ROOT + 'y_test.npy')
    print(X_train.shape, y_train.shape)
  ─────────────────────────────────────────────────────
""")

Libraries ready.
Mounted at /content/drive
Drive mounted. Project folder: /content/drive/MyDrive/ECG_FPGA_Project

Folder structure:
  /content/drive/MyDrive/ECG_FPGA_Project/
  ├── phase1_outputs/   ← figures
  ├── checkpoints/      ← intermediate saves (resume support)
  └── *.npy             ← final prepared arrays
Configuration ready.
  CSV directory  : /content/drive/MyDrive/ECG_FPGA_Project/
  Drive root     : /content/drive/MyDrive/ECG_FPGA_Project
  SMOTE target   : 20,000 per class
  Val split      : 20%
Checkpoint helpers ready.

STEP 1 — Loading MIT-BIH CSV files
  Training CSV : (87554, 188)
  Testing  CSV : (21892, 188)

  Raw class distribution:
    Class 0 (N): 72,471  ████████████████████████████████████
    Class 1 (S):  2,223  █
    Class 2 (V):  5,788  ██
    Class 3 (F):    641  
    Class 4 (Q):  6,431  ███
  Saved figure → 01_raw_distribution.png

  Step 1 complete.

STEP 2 — Band-pass filter (0.5–40 Hz)
  Filtering 87,554 training beats ...
  Training filter done

In [ ]:
"""
=============================================================================
PHASE 2 — MODEL DESIGN & TRAINING
ECG Arrhythmia Classification for PYNQ-Z2 FPGA Deployment
=============================================================================

What this script does:
  Step 1 : Load prepared data from Google Drive (saved by Phase 1)
  Step 2 : Build Hybrid-Parallel 1D-CNN architecture (from PACAC paper)
  Step 3 : Train with proper callbacks (EarlyStopping, ModelCheckpoint)
  Step 4 : Plot training curves (accuracy + loss)
  Step 5 : Evaluate on test set — confusion matrix + per-class metrics
  Step 6 : Save final model to Google Drive (.h5 format for Phase 3)

RUNTIME : Switch to GPU (Runtime → Change runtime type → T4 GPU)
          Training takes ~30-60 min on T4, ~3-4 hours on CPU

KEY IMPROVEMENTS over original GitHub code:
  ✓ Hybrid-Parallel architecture (not sequential)
  ✓ Dropout correctly connected to graph (original code had this bug)
  ✓ EarlyStopping — stops before overfitting
  ✓ ModelCheckpoint — saves best weights, not last weights
  ✓ Proper validation data (not test set)
  ✓ Full evaluation: F1, precision, recall per class
=============================================================================
"""


from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import os

# ── Same folder as Phase 1 ─────────────────────────────────────────────────
DRIVE_ROOT = '/content/drive/MyDrive/ECG_FPGA_Project'
MODEL_DIR  = os.path.join(DRIVE_ROOT, 'models')
FIG_DIR    = os.path.join(DRIVE_ROOT, 'phase2_outputs')

for d in [MODEL_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Load prepared arrays ───────────────────────────────────────────────────
print("Loading data from Google Drive ...")

X_train = np.load(os.path.join(DRIVE_ROOT, 'X_train.npy'))
y_train = np.load(os.path.join(DRIVE_ROOT, 'y_train.npy'))
X_val   = np.load(os.path.join(DRIVE_ROOT, 'X_val.npy'))
y_val   = np.load(os.path.join(DRIVE_ROOT, 'y_val.npy'))
X_test  = np.load(os.path.join(DRIVE_ROOT, 'X_test.npy'))
y_test  = np.load(os.path.join(DRIVE_ROOT, 'y_test.npy'))

print(f"  X_train : {X_train.shape}   y_train : {y_train.shape}")
print(f"  X_val   : {X_val.shape}   y_val   : {y_val.shape}")
print(f"  X_test  : {X_test.shape}  y_test  : {y_test.shape}")
print("Data loaded successfully.")



import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import (EarlyStopping, ModelCheckpoint,
                                        ReduceLROnPlateau, CSVLogger)
from tensorflow.keras.utils import to_categorical
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# ── GPU check ─────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
print(f"\nGPU available: {len(gpus) > 0}")
if gpus:
    print(f"  Device: {gpus[0].name}")
else:
    print("  WARNING: No GPU found. Training will be slow.")
    print("  Go to Runtime → Change runtime type → T4 GPU")

print(f"TensorFlow version: {tf.__version__}")



NUM_CLASSES  = 5
CLASS_SHORT  = ['N', 'S', 'V', 'F', 'Q']
CLASS_NAMES  = ['Normal', 'Supraventricular', 'Ventricular', 'Fusion', 'Unknown']

y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_val_cat   = to_categorical(y_val,   NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  NUM_CLASSES)

print("Labels one-hot encoded:")
print(f"  y_train_cat : {y_train_cat.shape}   example: {y_train_cat[0]}")
print(f"  y_val_cat   : {y_val_cat.shape}")
print(f"  y_test_cat  : {y_test_cat.shape}")


# Architecture based on PACAC paper (Mangaraj et al., GLSVLSI 2024)
#
# Structure per stage (repeated twice):
#   Sequential Conv → [Parallel Conv A ∥ Parallel Conv B]
#                   → Channel-wise addition → BatchNorm → ReLU → MaxPool
#
# Why parallel branches?
#   - Extracts features at different receptive fields simultaneously
#   - Maps naturally to FPGA parallel execution in Phase 4
#   - Reduces vanishing gradient vs purely sequential convolutions
#   - Fewer parameters than a deeper sequential network
#
# DROPOUT FIX:
#   Original GitHub code wrote:  drop = Dropout(0.2)   ← does NOTHING
#   This code correctly uses:    x = Dropout(0.2)(x)   ← actually applied

def build_hybrid_parallel_cnn(input_shape=(186, 1), num_classes=5):
    """
    Build the Hybrid-Parallel 1D-CNN for ECG arrhythmia classification.

    Args:
        input_shape : tuple — (timesteps, channels) = (186, 1)
        num_classes : int   — number of output classes = 5

    Returns:
        model : compiled Keras Model
    """

    inputs = Input(shape=input_shape, name='ecg_input')

    # ──────────────────────────────────────────────────────────────────────
    # STAGE 1
    # Sequential conv → parallel pair → add → BN → ReLU → MaxPool
    # Kernel=7, Filters=8 (lightweight for FPGA)
    # ──────────────────────────────────────────────────────────────────────
    # Sequential convolution (shared feature map)
    x = layers.Conv1D(8, kernel_size=7, padding='same',
                      use_bias=False, name='s1_conv_seq')(inputs)
    x = layers.BatchNormalization(name='s1_bn_seq')(x)
    x = layers.ReLU(name='s1_relu_seq')(x)

    # Parallel branch A
    a = layers.Conv1D(8, kernel_size=7, padding='same',
                      use_bias=False, name='s1_conv_A')(x)
    a = layers.BatchNormalization(name='s1_bn_A')(a)

    # Parallel branch B
    b = layers.Conv1D(8, kernel_size=7, padding='same',
                      use_bias=False, name='s1_conv_B')(x)
    b = layers.BatchNormalization(name='s1_bn_B')(b)

    # Channel-wise addition (merge branches)
    x = layers.Add(name='s1_add')([a, b])
    x = layers.ReLU(name='s1_relu_add')(x)
    x = layers.MaxPooling1D(pool_size=2, strides=2,
                             padding='same', name='s1_pool')(x)
    x = layers.Dropout(0.2, name='s1_drop')(x)       # ← CORRECTLY CONNECTED

    # ──────────────────────────────────────────────────────────────────────
    # STAGE 2
    # Same pattern, more filters (16) to capture higher-level features
    # ──────────────────────────────────────────────────────────────────────
    x = layers.Conv1D(16, kernel_size=7, padding='same',
                      use_bias=False, name='s2_conv_seq')(x)
    x = layers.BatchNormalization(name='s2_bn_seq')(x)
    x = layers.ReLU(name='s2_relu_seq')(x)

    # Parallel branch A
    a = layers.Conv1D(16, kernel_size=7, padding='same',
                      use_bias=False, name='s2_conv_A')(x)
    a = layers.BatchNormalization(name='s2_bn_A')(a)

    # Parallel branch B
    b = layers.Conv1D(16, kernel_size=7, padding='same',
                      use_bias=False, name='s2_conv_B')(x)
    b = layers.BatchNormalization(name='s2_bn_B')(b)

    x = layers.Add(name='s2_add')([a, b])
    x = layers.ReLU(name='s2_relu_add')(x)
    x = layers.MaxPooling1D(pool_size=2, strides=2,
                             padding='same', name='s2_pool')(x)
    x = layers.Dropout(0.2, name='s2_drop')(x)       # ← CORRECTLY CONNECTED

    # ──────────────────────────────────────────────────────────────────────
    # STAGE 3 — Sequential conv, 32 filters, smaller kernel
    # ──────────────────────────────────────────────────────────────────────
    x = layers.Conv1D(32, kernel_size=5, padding='same',
                      use_bias=False, name='s3_conv')(x)
    x = layers.BatchNormalization(name='s3_bn')(x)
    x = layers.ReLU(name='s3_relu')(x)
    x = layers.MaxPooling1D(pool_size=2, strides=2,
                             padding='same', name='s3_pool')(x)

    # ──────────────────────────────────────────────────────────────────────
    # STAGE 4 — Sequential conv, 64 filters
    # ──────────────────────────────────────────────────────────────────────
    x = layers.Conv1D(64, kernel_size=5, padding='same',
                      use_bias=False, name='s4_conv')(x)
    x = layers.BatchNormalization(name='s4_bn')(x)
    x = layers.ReLU(name='s4_relu')(x)
    x = layers.MaxPooling1D(pool_size=2, strides=2,
                             padding='same', name='s4_pool')(x)

    # ──────────────────────────────────────────────────────────────────────
    # CLASSIFIER HEAD
    # GlobalAveragePooling instead of Flatten — reduces parameters,
    # less prone to overfitting, easier to quantize for FPGA
    # ──────────────────────────────────────────────────────────────────────
    x = layers.GlobalAveragePooling1D(name='gap')(x)

    x = layers.Dense(64, name='fc1')(x)
    x = layers.BatchNormalization(name='fc1_bn')(x)
    x = layers.ReLU(name='fc1_relu')(x)
    x = layers.Dropout(0.3, name='fc1_drop')(x)       # ← CORRECTLY CONNECTED

    x = layers.Dense(32, name='fc2')(x)
    x = layers.BatchNormalization(name='fc2_bn')(x)
    x = layers.ReLU(name='fc2_relu')(x)
    x = layers.Dropout(0.2, name='fc2_drop')(x)       # ← CORRECTLY CONNECTED

    outputs = layers.Dense(num_classes, activation='softmax',
                           name='output')(x)

    model = Model(inputs=inputs, outputs=outputs, name='Hybrid_Parallel_1D_CNN')
    return model


# ── Build and compile ──────────────────────────────────────────────────────
model = build_hybrid_parallel_cnn(
    input_shape=(X_train.shape[1], X_train.shape[2]),
    num_classes=NUM_CLASSES
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ── Model summary ──────────────────────────────────────────────────────────
model.summary()

total_params = model.count_params()
print(f"\nTotal trainable parameters : {total_params:,}")
print(f"Target from PACAC paper    : ~24,865")
print(f"Lightweight for FPGA       : {'✓ YES' if total_params < 50000 else '✗ Too large'}")



def print_architecture_diagram():
    """Print a text diagram of the hybrid parallel architecture."""
    print("""
  ┌─────────────────────────────────────────────┐
  │          INPUT  (186 × 1)                   │
  └───────────────────┬─────────────────────────┘
                      │
  ┌───────────────────▼─────────────────────────┐
  │   STAGE 1 — Sequential Conv1D (K=7, F=8)   │
  │            BatchNorm + ReLU                 │
  └──────────┬──────────────────┬───────────────┘
             │                  │
  ┌──────────▼──────┐  ┌────────▼────────┐
  │ Parallel Conv A │  │ Parallel Conv B │   ← runs simultaneously
  │  (K=7, F=8)     │  │  (K=7, F=8)    │      on FPGA
  │  BatchNorm      │  │  BatchNorm      │
  └──────────┬──────┘  └────────┬────────┘
             └────────┬─────────┘
                      │  Channel-wise ADD
  ┌───────────────────▼─────────────────────────┐
  │         ReLU → MaxPool(2) → Dropout(0.2)   │
  └───────────────────┬─────────────────────────┘
                      │
  ┌───────────────────▼─────────────────────────┐
  │   STAGE 2 — Sequential Conv1D (K=7, F=16)  │
  │   + Parallel pair (F=16) + Add              │
  │   + ReLU → MaxPool(2) → Dropout(0.2)       │
  └───────────────────┬─────────────────────────┘
                      │
  ┌───────────────────▼─────────────────────────┐
  │   STAGE 3 — Conv1D (K=5, F=32)             │
  │             BatchNorm + ReLU + MaxPool(2)   │
  └───────────────────┬─────────────────────────┘
                      │
  ┌───────────────────▼─────────────────────────┐
  │   STAGE 4 — Conv1D (K=5, F=64)             │
  │             BatchNorm + ReLU + MaxPool(2)   │
  └───────────────────┬─────────────────────────┘
                      │
  ┌───────────────────▼─────────────────────────┐
  │   Global Average Pooling                    │
  └───────────────────┬─────────────────────────┘
                      │
  ┌───────────────────▼─────────────────────────┐
  │   Dense(64) + BN + ReLU + Dropout(0.3)     │
  └───────────────────┬─────────────────────────┘
                      │
  ┌───────────────────▼─────────────────────────┐
  │   Dense(32) + BN + ReLU + Dropout(0.2)     │
  └───────────────────┬─────────────────────────┘
                      │
  ┌───────────────────▼─────────────────────────┐
  │   OUTPUT — Dense(5, Softmax)                │
  │   N    S    V    F    Q                     │
  └─────────────────────────────────────────────┘
    """)

print_architecture_diagram()



BEST_MODEL_PATH = os.path.join(MODEL_DIR, 'best_model.h5')
FINAL_MODEL_PATH = os.path.join(MODEL_DIR, 'final_model.h5')
CSV_LOG_PATH    = os.path.join(MODEL_DIR, 'training_log.csv')

callbacks = [

    # ── Save best model (lowest val_loss) ─────────────────────────────────
    ModelCheckpoint(
        filepath=BEST_MODEL_PATH,
        monitor='val_loss',
        mode='min',
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    ),

    # ── Stop training when val_loss stops improving ────────────────────────
    # patience=20 means: stop if no improvement for 20 consecutive epochs
    EarlyStopping(
        monitor='val_loss',
        mode='min',
        patience=20,
        restore_best_weights=True,   # revert to best weights when stopping
        verbose=1
    ),

    # ── Reduce LR when training plateaus ──────────────────────────────────
    # Halves the learning rate if val_loss doesn't improve for 8 epochs
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=8,
        min_lr=1e-6,
        verbose=1
    ),

    # ── Save training log to Drive ────────────────────────────────────────
    CSVLogger(CSV_LOG_PATH, append=True)   # append=True survives interrupts
]

print("Callbacks configured:")
print(f"  ModelCheckpoint → {BEST_MODEL_PATH}")
print(f"  EarlyStopping   → patience=20, monitors val_loss")
print(f"  ReduceLROnPlateau → factor=0.5, patience=8")
print(f"  CSVLogger       → {CSV_LOG_PATH}")


# Training configuration:
#   Epochs    : 200 (EarlyStopping will stop it earlier if needed)
#   Batch size: 32  (good balance of speed and gradient stability)
#   Validation: X_val / y_val (NOT X_test — no data leakage)
#
# Expected training time:
#   T4 GPU  → ~30-60 minutes
#   CPU     → ~3-4 hours (not recommended)

EPOCHS     = 200
BATCH_SIZE = 32

print("=" * 60)
print("TRAINING START")
print("=" * 60)
print(f"  Epochs (max)  : {EPOCHS}")
print(f"  Batch size    : {BATCH_SIZE}")
print(f"  Training set  : {X_train.shape[0]:,} samples")
print(f"  Validation    : {X_val.shape[0]:,} samples")
print(f"  Best model    : {BEST_MODEL_PATH}")
print(f"  Early stop    : patience=20 on val_loss")
print()

history = model.fit(
    X_train, y_train_cat,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val_cat),   # ← proper validation, not test set
    callbacks=callbacks,
    verbose=1
)

print("\nTraining complete.")

# Save final model (regardless of whether it is the best)
model.save(FINAL_MODEL_PATH)
print(f"Final model saved → {FINAL_MODEL_PATH}")
print(f"Best model saved  → {BEST_MODEL_PATH}")



def save_fig(fig, filename):
    path = os.path.join(FIG_DIR, filename)
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved → {filename}")


hist = history.history
epochs_ran = len(hist['accuracy'])
epoch_axis = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Hybrid-Parallel 1D-CNN — Training Curves', fontsize=13, fontweight='bold')

# ── Accuracy ──────────────────────────────────────────────────────────────
axes[0].plot(epoch_axis, hist['accuracy'],     label='Train',      color='#4472C4', lw=1.5)
axes[0].plot(epoch_axis, hist['val_accuracy'], label='Validation', color='#ED7D31', lw=1.5)
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0.7, 1.01)

# Mark best epoch (lowest val_loss)
best_epoch = np.argmin(hist['val_loss']) + 1
axes[0].axvline(best_epoch, color='green', lw=1, ls='--', alpha=0.7, label=f'Best epoch ({best_epoch})')
axes[0].legend(loc='lower right')

# ── Loss ──────────────────────────────────────────────────────────────────
axes[1].plot(epoch_axis, hist['loss'],     label='Train',      color='#4472C4', lw=1.5)
axes[1].plot(epoch_axis, hist['val_loss'], label='Validation', color='#ED7D31', lw=1.5)
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss (Categorical Crossentropy)')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)
axes[1].axvline(best_epoch, color='green', lw=1, ls='--', alpha=0.7, label=f'Best epoch ({best_epoch})')
axes[1].legend(loc='upper right')

plt.tight_layout()
save_fig(fig, '01_training_curves.png')

# Print final numbers
print(f"\nTraining summary:")
print(f"  Epochs ran        : {epochs_ran} / {EPOCHS}")
print(f"  Best epoch        : {best_epoch}")
print(f"  Best val_accuracy : {max(hist['val_accuracy']):.4f}  ({max(hist['val_accuracy'])*100:.2f}%)")
print(f"  Best val_loss     : {min(hist['val_loss']):.4f}")
print(f"  Final train_acc   : {hist['accuracy'][-1]:.4f}")


# IMPORTANT: This is the FIRST time we ever use X_test.
# We load the BEST model (saved by ModelCheckpoint), not the final epoch.

print("\n" + "=" * 60)
print("EVALUATION ON TEST SET (using best saved model)")
print("=" * 60)

# Load best model
best_model = keras.models.load_model(BEST_MODEL_PATH)
print(f"Loaded: {BEST_MODEL_PATH}")

# Evaluate
test_loss, test_acc = best_model.evaluate(X_test, y_test_cat, verbose=0)
print(f"\n  Test Accuracy : {test_acc*100:.2f}%")
print(f"  Test Loss     : {test_loss:.4f}")

# Get predictions
y_pred_prob = best_model.predict(X_test, verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)
y_true      = y_test    # integer labels

# Per-class report
print("\n  Per-class classification report:")
print("  " + "-" * 60)
report = classification_report(
    y_true, y_pred,
    target_names=CLASS_NAMES,
    digits=4
)
for line in report.split('\n'):
    print("  " + line)

# Key metrics
f1_weighted = f1_score(y_true, y_pred, average='weighted')
f1_macro    = f1_score(y_true, y_pred, average='macro')
print(f"\n  F1 Weighted : {f1_weighted:.4f}")
print(f"  F1 Macro    : {f1_macro:.4f}")



cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]   # row-normalised

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Hybrid-Parallel 1D-CNN — Confusion Matrix on Test Set',
             fontsize=13, fontweight='bold')

# ── Raw counts ────────────────────────────────────────────────────────────
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_SHORT, yticklabels=CLASS_SHORT,
            ax=axes[0], linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[0].set_title('Raw counts')
axes[0].set_xlabel('Predicted label')
axes[0].set_ylabel('True label')

# ── Normalised ────────────────────────────────────────────────────────────
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=CLASS_SHORT, yticklabels=CLASS_SHORT,
            ax=axes[1], linewidths=0.5, cbar_kws={'shrink': 0.8},
            vmin=0, vmax=1)
axes[1].set_title('Normalised (per-class recall)')
axes[1].set_xlabel('Predicted label')
axes[1].set_ylabel('True label')

plt.tight_layout()
save_fig(fig, '02_confusion_matrix.png')

# Per-class accuracy from diagonal
print("\n  Per-class accuracy (from normalised confusion matrix diagonal):")
print("  " + "-" * 35)
for i, cls in enumerate(CLASS_SHORT):
    print(f"    {cls} ({CLASS_NAMES[i]:<17}) : {cm_norm[i,i]*100:6.2f}%")



f1_per_class = f1_score(y_true, y_pred, average=None)
pr_per_class = precision_score(y_true, y_pred, average=None)
rc_per_class = recall_score(y_true, y_pred, average=None)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(NUM_CLASSES)
w = 0.25

bars1 = ax.bar(x - w,   pr_per_class, w, label='Precision', color='#4472C4', alpha=0.85)
bars2 = ax.bar(x,       rc_per_class, w, label='Recall',    color='#ED7D31', alpha=0.85)
bars3 = ax.bar(x + w,   f1_per_class, w, label='F1-Score',  color='#A9D18E', alpha=0.85)

ax.set_title('Per-class Precision, Recall, F1-Score', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'{s}\n({n})' for s, n in zip(CLASS_SHORT, CLASS_NAMES)])
ax.set_ylabel('Score')
ax.set_ylim(0, 1.15)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)

# Value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                f'{h:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
save_fig(fig, '03_per_class_metrics.png')



results_path = os.path.join(MODEL_DIR, 'evaluation_results.txt')
with open(results_path, 'w') as f:
    f.write("=" * 60 + "\n")
    f.write("PHASE 2 — EVALUATION RESULTS\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Test Accuracy  : {test_acc*100:.4f}%\n")
    f.write(f"Test Loss      : {test_loss:.4f}\n")
    f.write(f"F1 Weighted    : {f1_weighted:.4f}\n")
    f.write(f"F1 Macro       : {f1_macro:.4f}\n")
    f.write(f"Epochs trained : {epochs_ran}\n")
    f.write(f"Best epoch     : {best_epoch}\n\n")
    f.write(classification_report(y_true, y_pred,
                                  target_names=CLASS_NAMES, digits=4))
    f.write("\nPer-class metrics:\n")
    f.write(f"{'Class':<20} {'Precision':>10} {'Recall':>10} {'F1':>10}\n")
    f.write("-" * 55 + "\n")
    for i, cls in enumerate(CLASS_NAMES):
        f.write(f"{cls:<20} {pr_per_class[i]:>10.4f} "
                f"{rc_per_class[i]:>10.4f} {f1_per_class[i]:>10.4f}\n")

print(f"Evaluation results saved → {results_path}")



print("\n" + "=" * 60)
print("PHASE 2 COMPLETE — SUMMARY")
print("=" * 60)

print(f"""
  Model         : Hybrid-Parallel 1D-CNN
  Parameters    : {total_params:,}
  Epochs ran    : {epochs_ran} / {EPOCHS}
  Best epoch    : {best_epoch}

  Results:
    Test accuracy  : {test_acc*100:.2f}%
    F1 weighted    : {f1_weighted:.4f}
    F1 macro       : {f1_macro:.4f}

  Files saved to {MODEL_DIR}:
    best_model.h5            ← USE THIS in Phase 3 (quantization)
    final_model.h5           ← last epoch weights
    training_log.csv         ← epoch-by-epoch history
    evaluation_results.txt   ← full metrics report

  Files saved to {FIG_DIR}:
    01_training_curves.png
    02_confusion_matrix.png
    03_per_class_metrics.png

  ─────────────────────────────────────────────────────────
  NEXT STEP — Phase 3: INT8 Quantization
  Load best_model.h5 and convert to INT8 using TFLite.
  Extract weights as C++ header files for Vitis HLS.
  ─────────────────────────────────────────────────────────
""")

Mounted at /content/drive
Loading data from Google Drive ...
  X_train : (80000, 186, 1)   y_train : (80000,)
  X_val   : (20000, 186, 1)   y_val   : (20000,)
  X_test  : (21892, 186, 1)  y_test  : (21892,)
Data loaded successfully.

GPU available: True
  Device: /physical_device:GPU:0
TensorFlow version: 2.19.0
Labels one-hot encoded:
  y_train_cat : (80000, 5)   example: [0. 0. 0. 0. 1.]
  y_val_cat   : (20000, 5)
  y_test_cat  : (21892, 5)


Model: "Hybrid_Parallel_1D_CNN"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ ecg_input           │ (None, 186, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s1_conv_seq         │ (None, 186, 8)    │         56 │ ecg_input[0][0]   │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s1_bn_seq           │ (None, 186, 8)    │         32 │ s1_conv_seq[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s1_relu_seq (ReLU)  │ (None, 186, 8)    │          0 │ s1_bn_seq[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s1_conv_A (Conv1D)  │ (None, 186, 8)    │        448 │ s1_relu_seq[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s1_conv_B (Conv1D)  │ (None, 186, 8)    │        448 │ s1_relu_seq[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s1_bn_A             │ (None, 186, 8)    │         32 │ s1_conv_A[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s1_bn_B             │ (None, 186, 8)    │         32 │ s1_conv_B[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s1_add (Add)        │ (None, 186, 8)    │          0 │ s1_bn_A[0][0],    │
│                     │                   │            │ s1_bn_B[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s1_relu_add (ReLU)  │ (None, 186, 8)    │          0 │ s1_add[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s1_pool             │ (None, 93, 8)     │          0 │ s1_relu_add[0][0] │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s1_drop (Dropout)   │ (None, 93, 8)     │          0 │ s1_pool[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s2_conv_seq         │ (None, 93, 16)    │        896 │ s1_drop[0][0]     │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s2_bn_seq           │ (None, 93, 16)    │         64 │ s2_conv_seq[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s2_relu_seq (ReLU)  │ (None, 93, 16)    │          0 │ s2_bn_seq[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s2_conv_A (Conv1D)  │ (None, 93, 16)    │      1,792 │ s2_relu_seq[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s2_conv_B (Conv1D)  │ (None, 93, 16)    │      1,792 │ s2_relu_seq[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s2_bn_A             │ (None, 93, 16)    │         64 │ s2_conv_A[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s2_bn_B             │ (None, 93, 16)    │         64 │ s2_conv_B[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 25,693 (100.36 KB)

 Trainable params: 25,165 (98.30 KB)

...


Epoch 56: finished saving model to /content/drive/MyDrive/ECG_FPGA_Project/models/best_model.h5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 14s 6ms/step - accuracy: 0.9785 - loss: 0.0633 - val_accuracy: 0.9783 - val_loss: 0.0592 - learning_rate: 6.2500e-05
Epoch 57/200
2494/2500 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9783 - loss: 0.0643
Epoch 57: val_loss did not improve from 0.05922
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.9789 - loss: 0.0626 - val_accuracy: 0.9767 - val_loss: 0.0659 - learning_rate: 6.2500e-05
Epoch 58/200
2499/2500 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9789 - loss: 0.0636
Epoch 58: val_loss did not improve from 0.05922
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.9793 - loss: 0.0619 - val_accuracy: 0.9769 - val_loss: 0.0636 - learning_rate: 6.2500e-05
Epoch 59/200
2495/2500 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9789 - loss: 0.0638
Epoch 59: val_loss did not improve from 0.05922
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step -


Epoch 63: finished saving model to /content/drive/MyDrive/ECG_FPGA_Project/models/best_model.h5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.9792 - loss: 0.0607 - val_accuracy: 0.9789 - val_loss: 0.0583 - learning_rate: 6.2500e-05
Epoch 64/200
2491/2500 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9801 - loss: 0.0599
Epoch 64: val_loss improved from 0.05829 to 0.05435, saving model to /content/drive/MyDrive/ECG_FPGA_Project/models/best_model.h5



Epoch 64: finished saving model to /content/drive/MyDrive/ECG_FPGA_Project/models/best_model.h5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.9796 - loss: 0.0607 - val_accuracy: 0.9808 - val_loss: 0.0543 - learning_rate: 6.2500e-05
Epoch 65/200
2496/2500 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9801 - loss: 0.0608
Epoch 65: val_loss did not improve from 0.05435
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.9797 - loss: 0.0606 - val_accuracy: 0.9774 - val_loss: 0.0638 - learning_rate: 6.2500e-05
Epoch 66/200
2499/2500 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9789 - loss: 0.0647
Epoch 66: val_loss did not improve from 0.05435
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.9790 - loss: 0.0637 - val_accuracy: 0.9776 - val_loss: 0.0640 - learning_rate: 6.2500e-05
Epoch 67/200
2493/2500 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9799 - loss: 0.0595
Epoch 67: val_loss did not improve from 0.05435
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step -


Training complete.
Final model saved → /content/drive/MyDrive/ECG_FPGA_Project/models/final_model.h5
Best model saved  → /content/drive/MyDrive/ECG_FPGA_Project/models/best_model.h5
  Saved → 01_training_curves.png

Training summary:
  Epochs ran        : 84 / 200
  Best epoch        : 64
  Best val_accuracy : 0.9808  (98.08%)
  Best val_loss     : 0.0543
  Final train_acc   : 0.9801

EVALUATION ON TEST SET (using best saved model)


Loaded: /content/drive/MyDrive/ECG_FPGA_Project/models/best_model.h5

  Test Accuracy : 93.65%
  Test Loss     : 0.1826

  Per-class classification report:
  ------------------------------------------------------------
                    precision    recall  f1-score   support
  
            Normal     0.9967    0.9306    0.9625     18118
  Supraventricular     0.4267    0.9155    0.5820       556
       Ventricular     0.9166    0.9565    0.9361      1448
            Fusion     0.2463    0.9136    0.3879       162
           Unknown     0.9575    0.9944    0.9756      1608
  
          accuracy                         0.9365     21892
         macro avg     0.7087    0.9421    0.7688     21892
      weighted avg     0.9685    0.9365    0.9478     21892
  

  F1 Weighted : 0.9478
  F1 Macro    : 0.7688
  Saved → 02_confusion_matrix.png

  Per-class accuracy (from normalised confusion matrix diagonal):
  -----------------------------------
    N (Normal           ) :  93.06%
    S (Sup

In [ ]:
"""
=============================================================================
PHASE 2 FIX — Improve S and F Class Precision
ECG Arrhythmia Classification for PYNQ-Z2 FPGA Deployment
=============================================================================

PROBLEM:
  S class precision : 0.4267  (model cries wolf too often for S beats)
  F class precision : 0.2463  (model cries wolf too often for F beats)
  Root cause: SMOTE created borderline synthetic samples too close to
              class N boundaries, confusing the model.

THREE FIXES APPLIED:
  Fix 1 — SMOTE+ENN instead of plain SMOTE
           ENN removes noisy/borderline samples after oversampling.
           Result: cleaner class boundaries.

  Fix 2 — Class weights during training
           Tells the model: "mistakes on S and F cost more than on N".
           Result: model stops being lazy about rare classes.

  Fix 3 — Label smoothing instead of hard one-hot targets
           Prevents the model from being overconfident.
           Result: better calibrated softmax outputs.

RUNTIME: GPU (T4) — Runtime → Change runtime type → T4 GPU
TIME:    ~45-60 minutes total (SMOTE+ENN ~20 min, training ~30 min)

WHAT IS REUSED FROM PHASE 1 (no re-running needed):
  ✓ Band-pass filtered arrays  (from Drive checkpoints)
  ✓ Test set                   (untouched)

WHAT IS REPLACED:
  ✗ Old balanced arrays (X_balanced, y_balanced) — overwritten
  ✗ Old model (best_model.h5)                    — overwritten
=============================================================================
"""


!pip install -q imbalanced-learn tensorflow-addons

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np

# ── Same Drive folder as before ────────────────────────────────────────────
DRIVE_ROOT     = '/content/drive/MyDrive/ECG_FPGA_Project'
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, 'checkpoints')
MODEL_DIR      = os.path.join(DRIVE_ROOT, 'models')
FIG_DIR        = os.path.join(DRIVE_ROOT, 'phase2_fix_outputs')

for d in [MODEL_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

print("Drive mounted.")
print(f"Project root : {DRIVE_ROOT}")



import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import (EarlyStopping, ModelCheckpoint,
                                        ReduceLROnPlateau, CSVLogger)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from imblearn.combine import SMOTEENN
from collections import Counter
import warnings, time
warnings.filterwarnings('ignore')

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# GPU check
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU available : {len(gpus) > 0}")
if gpus:
    print(f"  Device: {gpus[0].name}")

# Constants
NUM_CLASSES = 5
N_SAMPLES   = 186
VAL_SPLIT   = 0.20
CLASS_SHORT  = ['N', 'S', 'V', 'F', 'Q']
CLASS_NAMES  = ['Normal', 'Supraventricular', 'Ventricular', 'Fusion', 'Unknown']
CLASS_COLORS = ['#4472C4','#ED7D31','#A9D18E','#E74C3C','#7030A0']


# We reuse the band-pass filtered arrays saved in Phase 1.
# No need to re-filter — that step is already done.

print("\n" + "="*60)
print("STEP 1 — Loading filtered arrays from Phase 1 checkpoints")
print("="*60)

filt_train_path = os.path.join(CHECKPOINT_DIR, 'X_filtered_train.npy')
filt_test_path  = os.path.join(CHECKPOINT_DIR, 'X_filtered_test.npy')
y_raw_path      = os.path.join(CHECKPOINT_DIR, 'y_raw.npy')

# Check if intermediate files still exist
# (They may have been deleted by the cleanup cell in Phase 1)
if os.path.exists(filt_train_path):
    print("  Found Phase 1 checkpoint files. Loading ...")
    X_train_filt = np.load(filt_train_path)
    X_test_filt  = np.load(filt_test_path)
    y_raw        = np.load(y_raw_path)
    print(f"  X_train_filt : {X_train_filt.shape}")
    print(f"  X_test_filt  : {X_test_filt.shape}")
    print(f"  y_raw        : {y_raw.shape}")

else:
    # Checkpoint files were cleaned up — rebuild from the original CSVs
    # (This takes ~5 minutes)
    print("  Checkpoint files not found (were cleaned up).")
    print("  Rebuilding from CSV files ...")

    import pandas as pd
    from scipy.signal import butter, filtfilt

    CSV_DIR   = DRIVE_ROOT   # CSVs stored in same Drive folder
    TRAIN_CSV = os.path.join(CSV_DIR, 'mitbih_train.csv')
    TEST_CSV  = os.path.join(CSV_DIR, 'mitbih_test.csv')
    LABEL_COL = 187
    FS        = 125.0

    train_raw = pd.read_csv(TRAIN_CSV, header=None)
    test_raw  = pd.read_csv(TEST_CSV,  header=None)
    train_raw[LABEL_COL] = train_raw[LABEL_COL].astype(int)
    test_raw[LABEL_COL]  = test_raw[LABEL_COL].astype(int)

    X_raw  = train_raw.iloc[:, :N_SAMPLES].values.astype(np.float32)
    y_raw  = train_raw[LABEL_COL].values.astype(np.int64)
    X_test_raw = test_raw.iloc[:, :N_SAMPLES].values.astype(np.float32)
    y_test_raw = test_raw[LABEL_COL].values.astype(np.int64)

    def bandpass_filter(signal_1d, lowcut=0.5, highcut=40.0, fs=125.0, order=4):
        nyq  = 0.5 * fs
        b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
        return filtfilt(b, a, signal_1d).astype(np.float32)

    print(f"  Filtering {X_raw.shape[0]:,} training beats ...")
    t0 = time.time()
    X_train_filt = np.array([bandpass_filter(x) for x in X_raw])
    print(f"  Filtering {X_test_raw.shape[0]:,} test beats ...")
    X_test_filt  = np.array([bandpass_filter(x) for x in X_test_raw])
    print(f"  Done in {(time.time()-t0)/60:.1f} min.")

    # Re-save checkpoints
    np.save(filt_train_path, X_train_filt)
    np.save(filt_test_path,  X_test_filt)
    np.save(y_raw_path,      y_raw)
    print("  Checkpoint files re-saved.")

# Load test labels
y_test = np.load(os.path.join(DRIVE_ROOT, 'y_test.npy'))
print(f"\n  y_test : {y_test.shape}")
print("  Step 1 complete.")


# WHY SMOTE+ENN is better than plain SMOTE:
#
#   SMOTE creates synthetic samples by interpolating between real neighbors.
#   But some synthetics land in ambiguous regions (e.g. a synthetic S beat
#   that looks almost identical to a normal beat).
#
#   ENN (Edited Nearest Neighbors) cleans this up:
#   After SMOTE, ENN removes any sample whose class disagrees with the
#   majority class among its k nearest neighbors.
#   Result: cleaner, more separable class boundaries.
#
# NOTE: SMOTE+ENN does NOT guarantee exactly equal class sizes.
#       Class N will still be larger because it's cleaned less aggressively.
#       We handle this with class weights in training (Fix 2).

print("\n" + "="*60)
print("STEP 2 — FIX 1: SMOTE+ENN balancing")
print("="*60)

SMOTEENN_DONE = os.path.exists(
    os.path.join(CHECKPOINT_DIR, 'step_smoteenn.flag'))

if SMOTEENN_DONE:
    print("  ► Checkpoint found — loading SMOTE+ENN arrays ...")
    X_balanced = np.load(os.path.join(CHECKPOINT_DIR, 'X_smoteenn.npy'))
    y_balanced = np.load(os.path.join(CHECKPOINT_DIR, 'y_smoteenn.npy'))
    print(f"  X_balanced : {X_balanced.shape}")
    print(f"  Skipping SMOTE+ENN — already done.")

else:
    print("  Class distribution BEFORE SMOTE+ENN:")
    raw_counts = Counter(y_raw)
    for cls in range(5):
        print(f"    Class {cls} ({CLASS_SHORT[cls]}): {raw_counts[cls]:>6,}")

    print(f"\n  Running SMOTE+ENN (this takes ~15-25 min on CPU) ...")
    print("  You can leave this running safely.\n")

    t0 = time.time()

    # sampling_strategy: upsample all minority classes to match majority (N)
    # ENN will then clean borderline samples from all classes
    smote_enn = SMOTEENN(
        sampling_strategy='not majority',   # upsample all except class 0
        random_state=SEED,
        n_jobs=1                            # safe for all imblearn versions
    )

    X_balanced, y_balanced = smote_enn.fit_resample(X_train_filt, y_raw)
    elapsed = time.time() - t0
    print(f"  SMOTE+ENN completed in {elapsed/60:.1f} minutes.")

    print(f"\n  Class distribution AFTER SMOTE+ENN:")
    bal_counts = Counter(y_balanced)
    for cls in range(5):
        cnt = bal_counts[cls]
        bar = '█' * (cnt // 2000)
        print(f"    Class {cls} ({CLASS_SHORT[cls]}): {cnt:>6,}  {bar}")

    # Save checkpoint
    np.save(os.path.join(CHECKPOINT_DIR, 'X_smoteenn.npy'), X_balanced)
    np.save(os.path.join(CHECKPOINT_DIR, 'y_smoteenn.npy'), y_balanced)
    with open(os.path.join(CHECKPOINT_DIR, 'step_smoteenn.flag'), 'w') as f:
        f.write('done')
    print("\n  Checkpoint saved.")

    # Plot new distribution
    bal_counts = Counter(y_balanced)
    fig, ax = plt.subplots(figsize=(8, 4))
    vals = [bal_counts[i] for i in range(5)]
    ax.bar(CLASS_SHORT, vals, color=CLASS_COLORS, edgecolor='white', lw=0.8)
    ax.set_title('Class distribution after SMOTE+ENN', fontweight='bold')
    ax.set_xlabel('AAMI Class')
    ax.set_ylabel('Number of beats')
    for i, v in enumerate(vals):
        ax.text(i, v+200, f'{v:,}', ha='center', va='bottom', fontsize=9)
    fig.tight_layout()
    path = os.path.join(FIG_DIR, '01_smoteenn_distribution.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved → 01_smoteenn_distribution.png")

print("  Step 2 complete.")



print("\n" + "="*60)
print("STEP 3 — Train/val split + reshape")
print("="*60)

X_balanced, y_balanced = shuffle(X_balanced, y_balanced, random_state=SEED)

X_train, X_val, y_train, y_val = train_test_split(
    X_balanced, y_balanced,
    test_size=VAL_SPLIT,
    random_state=SEED,
    stratify=y_balanced
)

# Reshape (N, 186) → (N, 186, 1)
X_train      = X_train.reshape(-1, N_SAMPLES, 1).astype(np.float32)
X_val        = X_val.reshape(-1, N_SAMPLES, 1).astype(np.float32)
X_test_final = X_test_filt.reshape(-1, N_SAMPLES, 1).astype(np.float32)

print(f"  X_train      : {X_train.shape}   y_train : {y_train.shape}")
print(f"  X_val        : {X_val.shape}   y_val   : {y_val.shape}")
print(f"  X_test_final : {X_test_final.shape}  y_test  : {y_test.shape}")

# One-hot encode
from tensorflow.keras.utils import to_categorical
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_val_cat   = to_categorical(y_val,   NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  NUM_CLASSES)

print("  One-hot encoding done.")
print("  Step 3 complete.")


# Even after SMOTE+ENN, class N is still larger than S and F.
# Class weights tell the optimizer:
#   "a wrong prediction on class F should be penalised MORE
#    than a wrong prediction on class N"
# This forces the model to pay attention to rare classes.

print("\n" + "="*60)
print("STEP 4 — FIX 2: Computing class weights")
print("="*60)

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=y_train
)
class_weight_dict = {i: w for i, w in enumerate(class_weights_array)}

print(f"  Class weights (higher = model penalised more for mistakes):")
for cls in range(NUM_CLASSES):
    w = class_weight_dict[cls]
    bar = '█' * int(w * 10)
    print(f"    Class {cls} ({CLASS_SHORT[cls]}) : {w:.4f}  {bar}")

print("\n  Interpretation:")
print("  Classes S and F will have higher weights → model focuses on them more.")
print("  Step 4 complete.")


# Standard cross-entropy trains the model toward hard one-hot targets:
#   [0, 0, 1, 0, 0]  for class V
#
# Label smoothing softens these to:
#   [0.025, 0.025, 0.9, 0.025, 0.025]  (smooth=0.1)
#
# Why this helps:
#   - Prevents the model from being overconfident (very high softmax outputs)
#   - Reduces the tendency to memorise rare class patterns
#   - Improves calibration of probability outputs
#   - Particularly helps precision for S and F
#
# We use TF's built-in label smoothing — no extra library needed.

print("\n" + "="*60)
print("STEP 5 — FIX 3: Label smoothing loss")
print("="*60)

LABEL_SMOOTHING = 0.1   # standard value — smooths targets by 10%

loss_fn = keras.losses.CategoricalCrossentropy(
    label_smoothing=LABEL_SMOOTHING
)

print(f"  Loss function     : CategoricalCrossentropy")
print(f"  Label smoothing   : {LABEL_SMOOTHING}")
print(f"  Hard target  [V]  : [0.000, 0.000, 1.000, 0.000, 0.000]")
smoothed = [LABEL_SMOOTHING/NUM_CLASSES]*NUM_CLASSES
smoothed[2] = 1 - LABEL_SMOOTHING + LABEL_SMOOTHING/NUM_CLASSES
print(f"  Smooth target [V] : [{', '.join(f'{v:.3f}' for v in smoothed)}]")
print("  Step 5 complete.")



def build_hybrid_parallel_cnn(input_shape=(186, 1), num_classes=5):
    """
    Hybrid-Parallel 1D-CNN — same architecture as Phase 2.
    All three fixes are applied at the training level, not the architecture.
    Architecture is intentionally kept the same for fair comparison.
    """
    inputs = Input(shape=input_shape, name='ecg_input')

    # Stage 1 — K=7, F=8, parallel pair
    x = layers.Conv1D(8, 7, padding='same', use_bias=False, name='s1_conv_seq')(inputs)
    x = layers.BatchNormalization(name='s1_bn_seq')(x)
    x = layers.ReLU(name='s1_relu_seq')(x)

    a = layers.Conv1D(8, 7, padding='same', use_bias=False, name='s1_conv_A')(x)
    a = layers.BatchNormalization(name='s1_bn_A')(a)
    b = layers.Conv1D(8, 7, padding='same', use_bias=False, name='s1_conv_B')(x)
    b = layers.BatchNormalization(name='s1_bn_B')(b)

    x = layers.Add(name='s1_add')([a, b])
    x = layers.ReLU(name='s1_relu_add')(x)
    x = layers.MaxPooling1D(2, strides=2, padding='same', name='s1_pool')(x)
    x = layers.Dropout(0.2, name='s1_drop')(x)

    # Stage 2 — K=7, F=16, parallel pair
    x = layers.Conv1D(16, 7, padding='same', use_bias=False, name='s2_conv_seq')(x)
    x = layers.BatchNormalization(name='s2_bn_seq')(x)
    x = layers.ReLU(name='s2_relu_seq')(x)

    a = layers.Conv1D(16, 7, padding='same', use_bias=False, name='s2_conv_A')(x)
    a = layers.BatchNormalization(name='s2_bn_A')(a)
    b = layers.Conv1D(16, 7, padding='same', use_bias=False, name='s2_conv_B')(x)
    b = layers.BatchNormalization(name='s2_bn_B')(b)

    x = layers.Add(name='s2_add')([a, b])
    x = layers.ReLU(name='s2_relu_add')(x)
    x = layers.MaxPooling1D(2, strides=2, padding='same', name='s2_pool')(x)
    x = layers.Dropout(0.2, name='s2_drop')(x)

    # Stage 3 — K=5, F=32
    x = layers.Conv1D(32, 5, padding='same', use_bias=False, name='s3_conv')(x)
    x = layers.BatchNormalization(name='s3_bn')(x)
    x = layers.ReLU(name='s3_relu')(x)
    x = layers.MaxPooling1D(2, strides=2, padding='same', name='s3_pool')(x)

    # Stage 4 — K=5, F=64
    x = layers.Conv1D(64, 5, padding='same', use_bias=False, name='s4_conv')(x)
    x = layers.BatchNormalization(name='s4_bn')(x)
    x = layers.ReLU(name='s4_relu')(x)
    x = layers.MaxPooling1D(2, strides=2, padding='same', name='s4_pool')(x)

    # Classifier head
    x = layers.GlobalAveragePooling1D(name='gap')(x)
    x = layers.Dense(64, name='fc1')(x)
    x = layers.BatchNormalization(name='fc1_bn')(x)
    x = layers.ReLU(name='fc1_relu')(x)
    x = layers.Dropout(0.3, name='fc1_drop')(x)

    x = layers.Dense(32, name='fc2')(x)
    x = layers.BatchNormalization(name='fc2_bn')(x)
    x = layers.ReLU(name='fc2_relu')(x)
    x = layers.Dropout(0.2, name='fc2_drop')(x)

    outputs = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model = Model(inputs=inputs, outputs=outputs,
                  name='Hybrid_Parallel_1D_CNN_v2')
    return model


model = build_hybrid_parallel_cnn(
    input_shape=(X_train.shape[1], X_train.shape[2]),
    num_classes=NUM_CLASSES
)

# Compile with label smoothing loss
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=loss_fn,         # label smoothing applied here
    metrics=['accuracy']
)

print(f"Model built — parameters: {model.count_params():,}")



BEST_MODEL_PATH = os.path.join(MODEL_DIR, 'best_model.h5')   # overwrites old
CSV_LOG_PATH    = os.path.join(MODEL_DIR, 'training_log_v2.csv')

callbacks = [
    ModelCheckpoint(
        filepath=BEST_MODEL_PATH,
        monitor='val_loss',
        mode='min',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=8,
        min_lr=1e-6,
        verbose=1
    ),
    CSVLogger(CSV_LOG_PATH, append=False)
]

print("Callbacks ready.")



EPOCHS     = 200
BATCH_SIZE = 32

print("\n" + "="*60)
print("TRAINING START (v2 — with all fixes)")
print("="*60)
print(f"  Fix 1: SMOTE+ENN balanced data")
print(f"  Fix 2: Class weights applied")
print(f"  Fix 3: Label smoothing = {LABEL_SMOOTHING}")
print(f"  Epochs (max)  : {EPOCHS}")
print(f"  Batch size    : {BATCH_SIZE}")
print(f"  Training set  : {X_train.shape[0]:,}")
print(f"  Validation    : {X_val.shape[0]:,}")
print()

history = model.fit(
    X_train, y_train_cat,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weight_dict,     # Fix 2 applied here
    callbacks=callbacks,
    verbose=1
)

print("\nTraining complete.")
model.save(os.path.join(MODEL_DIR, 'final_model_v2.h5'))
print(f"Saved → final_model_v2.h5")
print(f"Saved → best_model.h5  (overwrites previous)")



def save_fig(fig, filename):
    path = os.path.join(FIG_DIR, filename)
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved → {filename}")


hist        = history.history
epochs_ran  = len(hist['accuracy'])
best_epoch  = int(np.argmin(hist['val_loss'])) + 1
epoch_axis  = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Hybrid-Parallel 1D-CNN v2 — Training Curves',
             fontsize=13, fontweight='bold')

axes[0].plot(epoch_axis, hist['accuracy'],     label='Train',      color='#4472C4', lw=1.5)
axes[0].plot(epoch_axis, hist['val_accuracy'], label='Validation', color='#ED7D31', lw=1.5)
axes[0].axvline(best_epoch, color='green', lw=1, ls='--', alpha=0.7,
                label=f'Best epoch ({best_epoch})')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0.7, 1.01)

axes[1].plot(epoch_axis, hist['loss'],     label='Train',      color='#4472C4', lw=1.5)
axes[1].plot(epoch_axis, hist['val_loss'], label='Validation', color='#ED7D31', lw=1.5)
axes[1].axvline(best_epoch, color='green', lw=1, ls='--', alpha=0.7,
                label=f'Best epoch ({best_epoch})')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
save_fig(fig, '02_training_curves_v2.png')

print(f"\n  Epochs ran    : {epochs_ran} / {EPOCHS}")
print(f"  Best epoch    : {best_epoch}")
print(f"  Best val_acc  : {max(hist['val_accuracy'])*100:.2f}%")
print(f"  Best val_loss : {min(hist['val_loss']):.4f}")



print("\n" + "="*60)
print("EVALUATION ON TEST SET")
print("="*60)

best_model  = keras.models.load_model(BEST_MODEL_PATH)
test_loss, test_acc = best_model.evaluate(X_test_final, y_test_cat, verbose=0)

print(f"\n  Test Accuracy : {test_acc*100:.2f}%")
print(f"  Test Loss     : {test_loss:.4f}")

y_pred_prob = best_model.predict(X_test_final, verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)
y_true      = y_test

print("\n  Per-class classification report:")
print("  " + "-"*62)
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4)
for line in report.split('\n'):
    print("  " + line)

f1_weighted = f1_score(y_true, y_pred, average='weighted')
f1_macro    = f1_score(y_true, y_pred, average='macro')
pr_classes  = precision_score(y_true, y_pred, average=None)
rc_classes  = recall_score(y_true, y_pred, average=None)
f1_classes  = f1_score(y_true, y_pred, average=None)



print("\n" + "="*60)
print("BEFORE vs AFTER COMPARISON")
print("="*60)

# Results from previous run (Phase 2 original)
prev_precision = [0.9967, 0.4267, 0.9166, 0.2463, 0.9575]
prev_recall    = [0.9306, 0.9155, 0.9565, 0.9136, 0.9944]
prev_f1        = [0.9625, 0.5820, 0.9361, 0.3879, 0.9756]
prev_acc       = 93.65

print(f"\n  {'Class':<22} {'── Before ──':^30} {'── After ──':^30}")
print(f"  {'':22} {'Prec':>8} {'Rec':>8} {'F1':>8}   {'Prec':>8} {'Rec':>8} {'F1':>8}")
print("  " + "-"*72)

for i in range(NUM_CLASSES):
    pchg = pr_classes[i] - prev_precision[i]
    fchg = f1_classes[i] - prev_f1[i]
    arrow_p = '↑' if pchg > 0.01 else ('↓' if pchg < -0.01 else '→')
    arrow_f = '↑' if fchg > 0.01 else ('↓' if fchg < -0.01 else '→')
    print(f"  {CLASS_NAMES[i]:<22} "
          f"{prev_precision[i]:>8.4f} {prev_recall[i]:>8.4f} {prev_f1[i]:>8.4f}   "
          f"{pr_classes[i]:>8.4f} {rc_classes[i]:>8.4f} {f1_classes[i]:>8.4f} "
          f" {arrow_p}{arrow_f}")

print(f"\n  Overall accuracy : {prev_acc:.2f}%  →  {test_acc*100:.2f}%")
print(f"  F1 weighted      : 0.9478   →  {f1_weighted:.4f}")
print(f"  F1 macro         : 0.7688   →  {f1_macro:.4f}")



cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Confusion Matrix — v2 (SMOTE+ENN + Class Weights + Label Smoothing)',
             fontsize=12, fontweight='bold')

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_SHORT, yticklabels=CLASS_SHORT,
            ax=axes[0], linewidths=0.5, cbar_kws={'shrink':0.8})
axes[0].set_title('Raw counts')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=CLASS_SHORT, yticklabels=CLASS_SHORT,
            ax=axes[1], linewidths=0.5, cbar_kws={'shrink':0.8},
            vmin=0, vmax=1)
axes[1].set_title('Normalised (per-class recall)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
save_fig(fig, '03_confusion_matrix_v2.png')



fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Before vs After — Per-class Precision, Recall, F1',
             fontsize=12, fontweight='bold')

x  = np.arange(NUM_CLASSES)
w  = 0.35
metric_pairs = [
    ('Precision', prev_precision, pr_classes),
    ('Recall',    prev_recall,    rc_classes),
    ('F1-Score',  prev_f1,        f1_classes),
]

for ax, (title, before, after) in zip(axes, metric_pairs):
    b1 = ax.bar(x - w/2, before, w, label='Before', color='#B0C4DE', edgecolor='white')
    b2 = ax.bar(x + w/2, after,  w, label='After',  color='#4472C4', edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_SHORT)
    ax.set_ylim(0, 1.2)
    ax.legend(fontsize=9)
    ax.grid(True, axis='y', alpha=0.3)
    for bar in b2:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                f'{h:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
save_fig(fig, '04_before_after_comparison.png')



print("\n" + "="*60)
print("SAVING UPDATED PREPARED DATA TO DRIVE")
print("="*60)

# These overwrite the Phase 1 .npy files with the SMOTE+ENN versions
# Phase 3 will use these new arrays
save_map = {
    'X_train.npy' : X_train,
    'y_train.npy' : y_train,
    'X_val.npy'   : X_val,
    'y_val.npy'   : y_val,
}
for fname, arr in save_map.items():
    path = os.path.join(DRIVE_ROOT, fname)
    np.save(path, arr)
    size = os.path.getsize(path) / 1024 / 1024
    print(f"  ✓ {fname:<18} {size:5.1f} MB   shape {arr.shape}")

# Save evaluation report
results_path = os.path.join(MODEL_DIR, 'evaluation_results_v2.txt')
with open(results_path, 'w') as f:
    f.write("PHASE 2 FIX — EVALUATION RESULTS\n")
    f.write("="*60 + "\n\n")
    f.write(f"Fixes applied:\n")
    f.write(f"  1. SMOTE+ENN balancing\n")
    f.write(f"  2. Class weights\n")
    f.write(f"  3. Label smoothing = {LABEL_SMOOTHING}\n\n")
    f.write(f"Test Accuracy  : {test_acc*100:.4f}%\n")
    f.write(f"Test Loss      : {test_loss:.4f}\n")
    f.write(f"F1 Weighted    : {f1_weighted:.4f}\n")
    f.write(f"F1 Macro       : {f1_macro:.4f}\n")
    f.write(f"Epochs trained : {epochs_ran}\n")
    f.write(f"Best epoch     : {best_epoch}\n\n")
    f.write(classification_report(y_true, y_pred,
                                  target_names=CLASS_NAMES, digits=4))
print(f"  ✓ evaluation_results_v2.txt saved")



print("\n" + "="*60)
print("PHASE 2 FIX COMPLETE — SUMMARY")
print("="*60)

print(f"""
  Three fixes applied:
    Fix 1 — SMOTE+ENN (cleaner class boundaries)
    Fix 2 — Class weights (focus on rare S and F)
    Fix 3 — Label smoothing = {LABEL_SMOOTHING} (better calibration)

  Results:
    Test accuracy  : {test_acc*100:.2f}%   (was 93.65%)
    F1 weighted    : {f1_weighted:.4f}  (was 0.9478)
    F1 macro       : {f1_macro:.4f}  (was 0.7688)

  Key class improvements:
    S precision    : {prev_precision[1]:.4f} → {pr_classes[1]:.4f}
    F precision    : {prev_precision[3]:.4f} → {pr_classes[3]:.4f}

  Files saved to Drive:
    models/best_model.h5              ← USE THIS in Phase 3
    models/evaluation_results_v2.txt
    phase2_fix_outputs/*.png

  ─────────────────────────────────────────────────────────
  NEXT STEP — Phase 3: INT8 Quantization
  Load models/best_model.h5 and convert weights to INT8
  for Vitis HLS C++ implementation.
  ─────────────────────────────────────────────────────────
""")

ERROR: Could not find a version that satisfies the requirement tensorflow-addons (from versions: none)
ERROR: No matching distribution found for tensorflow-addons
Mounted at /content/drive
Drive mounted.
Project root : /content/drive/MyDrive/ECG_FPGA_Project
GPU available : True
  Device: /physical_device:GPU:0

STEP 1 — Loading filtered arrays from Phase 1 checkpoints
  Checkpoint files not found (were cleaned up).
  Rebuilding from CSV files ...
  Filtering 87,554 training beats ...
  Filtering 21,892 test beats ...
  Done in 1.2 min.
  Checkpoint files re-saved.

  y_test : (21892,)
  Step 1 complete.

STEP 2 — FIX 1: SMOTE+ENN balancing
  Class distribution BEFORE SMOTE+ENN:
    Class 0 (N): 72,471
    Class 1 (S):  2,223
    Class 2 (V):  5,788
    Class 3 (F):    641
    Class 4 (Q):  6,431

  Running SMOTE+ENN (this takes ~15-25 min on CPU) ...
  You can leave this running safely.

  SMOTE+ENN completed in 29.6 minutes.

  Class distribution AFTER SMOTE+ENN:
    Class 0 (N): 69,


Epoch 1: finished saving model to /content/drive/MyDrive/ECG_FPGA_Project/models/best_model.h5
8973/8973 ━━━━━━━━━━━━━━━━━━━━ 71s 6ms/step - accuracy: 0.8802 - loss: 0.6629 - val_accuracy: 0.8716 - val_loss: 0.6451 - learning_rate: 0.0010
Epoch 2/200
8972/8973 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9342 - loss: 0.5555
Epoch 2: val_loss improved from 0.64509 to 0.56399, saving model to /content/drive/MyDrive/ECG_FPGA_Project/models/best_model.h5



Epoch 2: finished saving model to /content/drive/MyDrive/ECG_FPGA_Project/models/best_model.h5
8973/8973 ━━━━━━━━━━━━━━━━━━━━ 46s 5ms/step - accuracy: 0.9389 - loss: 0.5457 - val_accuracy: 0.9221 - val_loss: 0.5640 - learning_rate: 0.0010
Epoch 3/200
8968/8973 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9496 - loss: 0.5237
Epoch 3: val_loss improved from 0.56399 to 0.53687, saving model to /content/drive/MyDrive/ECG_FPGA_Project/models/best_model.h5



Epoch 3: finished saving model to /content/drive/MyDrive/ECG_FPGA_Project/models/best_model.h5
8973/8973 ━━━━━━━━━━━━━━━━━━━━ 47s 5ms/step - accuracy: 0.9514 - loss: 0.5201 - val_accuracy: 0.9308 - val_loss: 0.5369 - learning_rate: 0.0010
Epoch 4/200
8970/8973 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9566 - loss: 0.5098
Epoch 4: val_loss improved from 0.53687 to 0.50714, saving model to /content/drive/MyDrive/ECG_FPGA_Project/models/best_model.h5



Epoch 4: finished saving model to /content/drive/MyDrive/ECG_FPGA_Project/models/best_model.h5
8973/8973 ━━━━━━━━━━━━━━━━━━━━ 47s 5ms/step - accuracy: 0.9572 - loss: 0.5081 - val_accuracy: 0.9476 - val_loss: 0.5071 - learning_rate: 0.0010
Epoch 5/200
8963/8973 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9606 - loss: 0.5020
Epoch 5: val_loss improved from 0.50714 to 0.47020, saving model to /content/drive/MyDrive/ECG_FPGA_Project/models/best_model.h5


...


Epoch 64: finished saving model to /content/drive/MyDrive/ECG_FPGA_Project/models/best_model.h5
8973/8973 ━━━━━━━━━━━━━━━━━━━━ 53s 6ms/step - accuracy: 0.9852 - loss: 0.4497 - val_accuracy: 0.9890 - val_loss: 0.4155 - learning_rate: 3.1250e-05
Epoch 65/200
8962/8973 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9853 - loss: 0.4492
Epoch 65: val_loss did not improve from 0.41548
8973/8973 ━━━━━━━━━━━━━━━━━━━━ 47s 5ms/step - accuracy: 0.9853 - loss: 0.4493 - val_accuracy: 0.9885 - val_loss: 0.4166 - learning_rate: 3.1250e-05
Epoch 66/200
8969/8973 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9855 - loss: 0.4495
Epoch 66: val_loss improved from 0.41548 to 0.41473, saving model to /content/drive/MyDrive/ECG_FPGA_Project/models/best_model.h5



Epoch 66: finished saving model to /content/drive/MyDrive/ECG_FPGA_Project/models/best_model.h5
8973/8973 ━━━━━━━━━━━━━━━━━━━━ 48s 5ms/step - accuracy: 0.9855 - loss: 0.4498 - val_accuracy: 0.9893 - val_loss: 0.4147 - learning_rate: 3.1250e-05
Epoch 67/200
8971/8973 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9853 - loss: 0.4493
Epoch 67: val_loss did not improve from 0.41473
8973/8973 ━━━━━━━━━━━━━━━━━━━━ 46s 5ms/step - accuracy: 0.9855 - loss: 0.4494 - val_accuracy: 0.9889 - val_loss: 0.4160 - learning_rate: 3.1250e-05
Epoch 68/200
8964/8973 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9855 - loss: 0.4491
Epoch 68: val_loss did not improve from 0.41473
8973/8973 ━━━━━━━━━━━━━━━━━━━━ 47s 5ms/step - accuracy: 0.9854 - loss: 0.4496 - val_accuracy: 0.9887 - val_loss: 0.4163 - learning_rate: 3.1250e-05
Epoch 69/200
8971/8973 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9853 - loss: 0.4496
Epoch 69: val_loss did not improve from 0.41473
8973/8973 ━━━━━━━━━━━━━━━━━━━━ 47s 5ms/step -


Epoch 81: finished saving model to /content/drive/MyDrive/ECG_FPGA_Project/models/best_model.h5
8973/8973 ━━━━━━━━━━━━━━━━━━━━ 48s 5ms/step - accuracy: 0.9856 - loss: 0.4490 - val_accuracy: 0.9897 - val_loss: 0.4143 - learning_rate: 1.5625e-05
Epoch 82/200
8966/8973 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9858 - loss: 0.4483
Epoch 82: val_loss did not improve from 0.41430
8973/8973 ━━━━━━━━━━━━━━━━━━━━ 46s 5ms/step - accuracy: 0.9858 - loss: 0.4486 - val_accuracy: 0.9890 - val_loss: 0.4156 - learning_rate: 1.5625e-05
Epoch 83/200
8968/8973 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9856 - loss: 0.4488
Epoch 83: val_loss did not improve from 0.41430
8973/8973 ━━━━━━━━━━━━━━━━━━━━ 46s 5ms/step - accuracy: 0.9855 - loss: 0.4492 - val_accuracy: 0.9892 - val_loss: 0.4152 - learning_rate: 1.5625e-05
Epoch 84/200
8968/8973 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9857 - loss: 0.4481
Epoch 84: val_loss did not improve from 0.41430
8973/8973 ━━━━━━━━━━━━━━━━━━━━ 47s 5ms/step -


Training complete.
Saved → final_model_v2.h5
Saved → best_model.h5  (overwrites previous)
  Saved → 02_training_curves_v2.png

  Epochs ran    : 101 / 200
  Best epoch    : 81
  Best val_acc  : 98.97%
  Best val_loss : 0.4143

EVALUATION ON TEST SET



  Test Accuracy : 94.52%
  Test Loss     : 0.5111

  Per-class classification report:
  --------------------------------------------------------------
                    precision    recall  f1-score   support
  
            Normal     0.9967    0.9411    0.9681     18118
  Supraventricular     0.4669    0.9137    0.6180       556
       Ventricular     0.9115    0.9599    0.9351      1448
            Fusion     0.2986    0.9012    0.4485       162
           Unknown     0.9495    0.9938    0.9711      1608
  
          accuracy                         0.9452     21892
         macro avg     0.7246    0.9419    0.7882     21892
      weighted avg     0.9690    0.9452    0.9534     21892
  

BEFORE vs AFTER COMPARISON

  Class                           ── Before ──                   ── After ──          
                             Prec      Rec       F1       Prec      Rec       F1
  ------------------------------------------------------------------------
  Normal                   

In [ ]:
"""
=============================================================================
PHASE 3 — INT8 QUANTIZATION + C++ HEADER FILE EXTRACTION
ECG Arrhythmia Classification for PYNQ-Z2 FPGA Deployment
=============================================================================

What this script does:
  Step 1 : Load best_model.h5 from Google Drive
  Step 2 : Convert to TFLite INT8 (Post-Training Quantization)
  Step 3 : Verify quantized model accuracy (must be within 2% of original)
  Step 4 : Extract every layer's weights and biases as INT8 arrays
  Step 5 : Write C++ header files (.h) for each layer
  Step 6 : Write a master weights.h that includes everything
  Step 7 : Write model_config.h with architecture constants
  Step 8 : Verify header files are valid C++ syntax
  Step 9 : Package everything into a ZIP for Vitis HLS project

RUNTIME : CPU is fine — no GPU needed for this phase
          Takes ~5-10 minutes total

WHY INT8:
  - PYNQ-Z2 has 630KB BRAM and 220 DSP slices
  - Float32 model : each weight = 4 bytes
  - INT8 model    : each weight = 1 byte  (4x smaller)
  - INT8 on FPGA  : uses integer DSPs, not floating point units
  - Result        : fits in BRAM, runs faster, uses less power

OUTPUT FILES (go into Vitis HLS C++ project):
  weights/
  ├── model_config.h      ← architecture constants (kernel sizes, filter counts)
  ├── weights.h           ← master include file
  ├── s1_conv_seq_w.h     ← Stage 1 sequential conv weights
  ├── s1_conv_seq_b.h     ← Stage 1 sequential conv biases
  ├── s1_conv_A_w.h       ← Stage 1 parallel branch A weights
  ... (one file per layer)
  └── quantization_params.h  ← scale and zero_point for each layer
=============================================================================
"""


from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np

DRIVE_ROOT   = '/content/drive/MyDrive/ECG_FPGA_Project'
MODEL_DIR    = os.path.join(DRIVE_ROOT, 'models')
WEIGHTS_DIR  = os.path.join(DRIVE_ROOT, 'phase3_weights')
TFLITE_DIR   = os.path.join(DRIVE_ROOT, 'phase3_tflite')
FIG_DIR      = os.path.join(DRIVE_ROOT, 'phase3_outputs')

for d in [WEIGHTS_DIR, TFLITE_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

print("Drive mounted.")
print(f"Model dir   : {MODEL_DIR}")
print(f"Weights dir : {WEIGHTS_DIR}")
print(f"TFLite dir  : {TFLITE_DIR}")



import tensorflow as tf
from tensorflow import keras
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import warnings, zipfile, shutil, textwrap
warnings.filterwarnings('ignore')

print(f"TensorFlow version : {tf.__version__}")

# Constants
NUM_CLASSES = 5
N_SAMPLES   = 186
CLASS_SHORT = ['N', 'S', 'V', 'F', 'Q']
CLASS_NAMES = ['Normal', 'Supraventricular', 'Ventricular', 'Fusion', 'Unknown']



print("\n" + "="*60)
print("STEP 1 — Loading model and test data")
print("="*60)

best_model_path = os.path.join(MODEL_DIR, 'best_model.h5')
best_model      = keras.models.load_model(best_model_path)

print(f"  Model loaded : {best_model_path}")
print(f"  Parameters   : {best_model.count_params():,}")

# Load test data
X_test = np.load(os.path.join(DRIVE_ROOT, 'X_test.npy'))
y_test = np.load(os.path.join(DRIVE_ROOT, 'y_test.npy'))

print(f"  X_test       : {X_test.shape}   dtype: {X_test.dtype}")
print(f"  y_test       : {y_test.shape}")

# Quick baseline evaluation (float32 model)
print("\n  Evaluating float32 model on test set ...")
y_pred_float = np.argmax(best_model.predict(X_test, verbose=0), axis=1)
float_acc    = np.mean(y_pred_float == y_test)
print(f"  Float32 test accuracy : {float_acc*100:.2f}%")
print("  Step 1 complete.")


# HOW POST-TRAINING QUANTIZATION WORKS:
#
#   Float32 weight range  : e.g. [-0.342, +0.518]
#   INT8 range            : [-128, +127]
#
#   Quantization maps the float range to INT8:
#     scale     = (max - min) / 255
#     zero_point = round(-min / scale) - 128
#     int8_val  = round(float_val / scale) + zero_point
#
#   At inference time, INT8 multiply-accumulate operations are used.
#   Final output is dequantized back to float32 for the softmax.
#
# REPRESENTATIVE DATASET:
#   TFLite needs sample inputs to measure the actual activation ranges
#   during calibration. We use 200 random test samples — enough to
#   capture the full distribution without taking too long.

print("\n" + "="*60)
print("STEP 2 — TFLite INT8 Post-Training Quantization")
print("="*60)

# ── Build representative dataset generator ────────────────────────────────
N_CALIB = 200    # number of calibration samples

def representative_dataset():
    """
    Generator that yields calibration samples for TFLite INT8 quantization.
    TFLite uses these to compute activation ranges for each layer.
    """
    indices = np.random.choice(len(X_test), N_CALIB, replace=False)
    for i in indices:
        sample = X_test[i:i+1].astype(np.float32)    # shape (1, 186, 1)
        yield [sample]

# ── Create TFLite converter ────────────────────────────────────────────────
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)

# Full INT8 quantization (weights AND activations)
converter.optimizations          = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type   = tf.int8    # input tensor type
converter.inference_output_type  = tf.int8    # output tensor type

print(f"  Converting model to INT8 TFLite ...")
print(f"  Using {N_CALIB} calibration samples ...")

tflite_model = converter.convert()

# ── Save TFLite model ──────────────────────────────────────────────────────
tflite_path = os.path.join(TFLITE_DIR, 'model_int8.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

float_size  = os.path.getsize(best_model_path) / 1024
tflite_size = os.path.getsize(tflite_path)     / 1024

print(f"\n  Float32 model size : {float_size:.1f} KB")
print(f"  INT8 TFLite size   : {tflite_size:.1f} KB")
print(f"  Size reduction     : {float_size/tflite_size:.1f}x smaller")
print(f"  Saved              : {tflite_path}")
print("  Step 2 complete.")


# We must verify the INT8 model gives similar accuracy to the float model.
# Acceptable accuracy drop: < 2%
# If it drops more, we need to try quantization-aware training instead.

print("\n" + "="*60)
print("STEP 3 — Verify INT8 model accuracy")
print("="*60)

# Load TFLite interpreter
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"  Input  tensor : shape={input_details[0]['shape']}  "
      f"dtype={input_details[0]['dtype'].__name__}  "
      f"scale={input_details[0]['quantization'][0]:.6f}  "
      f"zero_point={input_details[0]['quantization'][1]}")
print(f"  Output tensor : shape={output_details[0]['shape']}  "
      f"dtype={output_details[0]['dtype'].__name__}  "
      f"scale={output_details[0]['quantization'][0]:.6f}  "
      f"zero_point={output_details[0]['quantization'][1]}")

# ── Run inference on full test set ────────────────────────────────────────
input_scale      = input_details[0]['quantization'][0]
input_zero_point = input_details[0]['quantization'][1]

print(f"\n  Running INT8 inference on {len(X_test):,} test samples ...")
print("  (This may take 3-5 minutes on CPU)")

y_pred_int8 = []
for i in range(len(X_test)):
    sample = X_test[i:i+1].astype(np.float32)

    # Quantize float32 input → int8
    sample_int8 = (sample / input_scale + input_zero_point).astype(np.int8)

    interpreter.set_tensor(input_details[0]['index'], sample_int8)
    interpreter.invoke()

    output = interpreter.get_tensor(output_details[0]['index'])  # int8 output
    y_pred_int8.append(np.argmax(output))

    if (i+1) % 5000 == 0:
        print(f"    {i+1:,} / {len(X_test):,} done ...")

y_pred_int8 = np.array(y_pred_int8)
int8_acc    = np.mean(y_pred_int8 == y_test)

print(f"\n  Float32 accuracy : {float_acc*100:.2f}%")
print(f"  INT8    accuracy : {int8_acc*100:.2f}%")
print(f"  Accuracy drop    : {(float_acc - int8_acc)*100:.2f}%")

if abs(float_acc - int8_acc) < 0.02:
    print("  ✓ PASS — accuracy drop < 2%, safe to proceed")
else:
    print("  ✗ WARNING — accuracy drop > 2%")
    print("    Consider quantization-aware training (QAT)")
    print("    For now, proceeding with current quantization")

print("\n  Per-class report (INT8 model):")
print("  " + "-"*55)
report = classification_report(y_test, y_pred_int8,
                                target_names=CLASS_NAMES, digits=4)
for line in report.split('\n'):
    print("  " + line)

print("  Step 3 complete.")


# We extract from the FLOAT model (not TFLite) using Keras layer API.
# Then we manually quantize to INT8 using asymmetric quantization.
#
# WHY NOT EXTRACT FROM TFLITE?
#   TFLite stores weights in a flatbuffer format that is hard to parse.
#   Keras gives clean numpy arrays per layer — much easier to work with.
#   We apply the same quantization formula manually.
#
# QUANTIZATION FORMULA (asymmetric):
#   scale      = (max - min) / (2^8 - 1)  = (max - min) / 255
#   zero_point = round(-min / scale) - 128
#   q          = clip(round(w / scale) + zero_point, -128, 127)
#   dequant    = (q - zero_point) * scale

print("\n" + "="*60)
print("STEP 4 — Extracting and quantizing layer weights")
print("="*60)

def quantize_to_int8(weights_float):
    """
    Quantize a float32 weight array to INT8 using asymmetric quantization.

    Args:
        weights_float : numpy array of float32 weights

    Returns:
        weights_int8  : numpy array of int8 weights
        scale         : float, quantization scale
        zero_point    : int, quantization zero point
    """
    w_min = float(weights_float.min())
    w_max = float(weights_float.max())

    # Avoid division by zero for zero-weight layers
    if w_max == w_min:
        return np.zeros_like(weights_float, dtype=np.int8), 1.0, 0

    scale      = (w_max - w_min) / 255.0
    zero_point = int(round(-w_min / scale)) - 128
    zero_point = np.clip(zero_point, -128, 127)

    # Quantize: float → int8
    w_quant = np.round(weights_float / scale) + zero_point
    w_int8  = np.clip(w_quant, -128, 127).astype(np.int8)

    return w_int8, scale, zero_point


def dequantize(weights_int8, scale, zero_point):
    """Reverse INT8 → float32 for verification."""
    return (weights_int8.astype(np.float32) - zero_point) * scale


# ── Walk through all layers and extract weights ────────────────────────────
layer_data = {}     # stores {layer_name: {weights, biases, quant params}}

print(f"  {'Layer':<30} {'Type':<20} {'W shape':<20} {'B shape'}")
print("  " + "-"*80)

for layer in best_model.layers:
    layer_weights = layer.get_weights()

    if len(layer_weights) == 0:
        continue    # skip layers with no weights (Pool, ReLU, Dropout, etc.)

    layer_name = layer.name

    # Conv1D layer: weights shape (kernel, in_channels, out_filters)
    # Dense layer:  weights shape (in_features, out_features)
    # BatchNorm:    gamma, beta, moving_mean, moving_var (4 arrays)

    if isinstance(layer, keras.layers.Conv1D):
        w_float = layer_weights[0]    # shape (kernel_size, 1, n_filters)
        b_float = layer_weights[1] if len(layer_weights) > 1 else None

        # Conv layers with use_bias=False have no bias array
        if layer.use_bias and b_float is not None:
            w_int8, w_scale, w_zp = quantize_to_int8(w_float)
            b_int8, b_scale, b_zp = quantize_to_int8(b_float)
        else:
            w_int8, w_scale, w_zp = quantize_to_int8(w_float)
            b_int8, b_scale, b_zp = None, None, None

        layer_data[layer_name] = {
            'type'    : 'Conv1D',
            'w_float' : w_float,
            'w_int8'  : w_int8,
            'w_scale' : w_scale,
            'w_zp'    : w_zp,
            'b_float' : b_float,
            'b_int8'  : b_int8,
            'b_scale' : b_scale,
            'b_zp'    : b_zp,
            'shape'   : w_float.shape,
        }
        b_shape = b_float.shape if b_float is not None else 'None'
        print(f"  {layer_name:<30} {'Conv1D':<20} {str(w_float.shape):<20} {b_shape}")

    elif isinstance(layer, keras.layers.Dense):
        w_float = layer_weights[0]
        b_float = layer_weights[1]

        w_int8, w_scale, w_zp = quantize_to_int8(w_float)
        b_int8, b_scale, b_zp = quantize_to_int8(b_float)

        layer_data[layer_name] = {
            'type'    : 'Dense',
            'w_float' : w_float,
            'w_int8'  : w_int8,
            'w_scale' : w_scale,
            'w_zp'    : w_zp,
            'b_float' : b_float,
            'b_int8'  : b_int8,
            'b_scale' : b_scale,
            'b_zp'    : b_zp,
            'shape'   : w_float.shape,
        }
        print(f"  {layer_name:<30} {'Dense':<20} {str(w_float.shape):<20} {str(b_float.shape)}")

    elif isinstance(layer, keras.layers.BatchNormalization):
        # BatchNorm has: gamma, beta, moving_mean, moving_var
        gamma  = layer_weights[0]
        beta   = layer_weights[1]
        m_mean = layer_weights[2]
        m_var  = layer_weights[3]

        layer_data[layer_name] = {
            'type'   : 'BatchNorm',
            'gamma'  : gamma.astype(np.float32),
            'beta'   : beta.astype(np.float32),
            'mean'   : m_mean.astype(np.float32),
            'var'    : m_var.astype(np.float32),
            'shape'  : gamma.shape,
        }
        print(f"  {layer_name:<30} {'BatchNorm':<20} {str(gamma.shape):<20} {str(beta.shape)}")

print(f"\n  Total weighted layers found: {len(layer_data)}")

# ── Verify quantization error is acceptable ───────────────────────────────
print("\n  Quantization error check (max absolute error per layer):")
print(f"  {'Layer':<30} {'Max Error':>12} {'Mean Error':>12} {'Status'}")
print("  " + "-"*65)

for name, d in layer_data.items():
    if d['type'] == 'BatchNorm':
        continue
    w_orig   = d['w_float']
    w_deq    = dequantize(d['w_int8'], d['w_scale'], d['w_zp'])
    max_err  = np.abs(w_orig - w_deq).max()
    mean_err = np.abs(w_orig - w_deq).mean()
    status   = '✓' if max_err < 0.01 else '⚠'
    print(f"  {name:<30} {max_err:>12.6f} {mean_err:>12.6f} {status}")

print("  Step 4 complete.")


# Each layer gets its own .h file containing:
#   - Weight array as int8_t (for Conv/Dense)
#   - Bias array as int32_t (biases kept in float for BatchNorm)
#   - Scale and zero_point as #define constants
#
# FORMAT FOR CONV WEIGHTS:
#   // s1_conv_seq_w.h
#   #pragma once
#   #include <cstdint>
#   #define S1_CONV_SEQ_W_KERNEL  7
#   #define S1_CONV_SEQ_W_IN_CH   1
#   #define S1_CONV_SEQ_W_OUT_CH  8
#   #define S1_CONV_SEQ_W_SCALE   0.003456f
#   #define S1_CONV_SEQ_W_ZP      -12
#   static const int8_t s1_conv_seq_w[7][1][8] = { ... };

print("\n" + "="*60)
print("STEP 5 — Writing C++ header files")
print("="*60)

def array_to_c_initializer(arr, indent=4):
    """
    Convert a numpy array of any shape to a nested C initializer list.
    e.g. [[1,2],[3,4]] → "{{1, 2}, {3, 4}}"
    """
    if arr.ndim == 1:
        vals = ', '.join(str(int(v)) for v in arr.flatten())
        return '{' + vals + '}'
    else:
        inner = [array_to_c_initializer(arr[i], indent) for i in range(arr.shape[0])]
        return '{' + ', '.join(inner) + '}'


def shape_to_c_dims(shape):
    """Convert numpy shape tuple to C array dimension string."""
    return ''.join(f'[{d}]' for d in shape)


def name_to_macro(layer_name, suffix):
    """Convert layer name to uppercase C macro name."""
    return (layer_name.upper().replace('-', '_') + '_' + suffix.upper())


all_header_files = []

for layer_name, d in layer_data.items():
    macro_base = layer_name.upper().replace('-', '_')
    ltype      = d['type']

    # ── Conv1D layer ──────────────────────────────────────────────────────
    if ltype == 'Conv1D':
        w       = d['w_int8']         # shape (kernel, in_ch, out_ch)
        w_scale = d['w_scale']
        w_zp    = d['w_zp']
        k, ic, oc = w.shape

        fname = f'{layer_name}_w.h'
        fpath = os.path.join(WEIGHTS_DIR, fname)

        lines = [
            f'// Auto-generated by Phase 3 quantization script',
            f'// Layer: {layer_name}  Type: Conv1D',
            f'// Weight shape: {w.shape}  (kernel, in_channels, out_filters)',
            f'',
            f'#pragma once',
            f'#include <cstdint>',
            f'',
            f'#define {macro_base}_KERNEL  {k}',
            f'#define {macro_base}_IN_CH   {ic}',
            f'#define {macro_base}_OUT_CH  {oc}',
            f'#define {macro_base}_W_SCALE {w_scale:.8f}f',
            f'#define {macro_base}_W_ZP    {w_zp}',
            f'',
            f'static const int8_t {layer_name}_w'
            f'[{k}][{ic}][{oc}] = ',
            array_to_c_initializer(w) + ';',
        ]

        with open(fpath, 'w') as f:
            f.write('\n'.join(lines))
        all_header_files.append(fname)
        print(f"  Written: {fname}   ({os.path.getsize(fpath)//1024} KB)")

        # Bias file (if layer has bias — our layers use use_bias=False for conv)
        if d['b_int8'] is not None:
            b     = d['b_int8']
            fname = f'{layer_name}_b.h'
            fpath = os.path.join(WEIGHTS_DIR, fname)
            lines = [
                f'#pragma once',
                f'#include <cstdint>',
                f'',
                f'#define {macro_base}_B_SCALE {d["b_scale"]:.8f}f',
                f'#define {macro_base}_B_ZP    {d["b_zp"]}',
                f'',
                f'static const int8_t {layer_name}_b[{len(b)}] = ',
                array_to_c_initializer(b) + ';',
            ]
            with open(fpath, 'w') as f:
                f.write('\n'.join(lines))
            all_header_files.append(fname)

    # ── Dense layer ───────────────────────────────────────────────────────
    elif ltype == 'Dense':
        w       = d['w_int8']      # shape (in_features, out_features)
        b       = d['b_int8']      # shape (out_features,)
        w_scale = d['w_scale']
        w_zp    = d['w_zp']
        inf, outf = w.shape

        fname = f'{layer_name}_w.h'
        fpath = os.path.join(WEIGHTS_DIR, fname)
        lines = [
            f'// Auto-generated — Layer: {layer_name}  Type: Dense',
            f'// Weight shape: {w.shape}  (in_features, out_features)',
            f'',
            f'#pragma once',
            f'#include <cstdint>',
            f'',
            f'#define {macro_base}_IN   {inf}',
            f'#define {macro_base}_OUT  {outf}',
            f'#define {macro_base}_W_SCALE {w_scale:.8f}f',
            f'#define {macro_base}_W_ZP    {w_zp}',
            f'',
            f'static const int8_t {layer_name}_w[{inf}][{outf}] = ',
            array_to_c_initializer(w) + ';',
        ]
        with open(fpath, 'w') as f:
            f.write('\n'.join(lines))
        all_header_files.append(fname)
        print(f"  Written: {fname}   ({os.path.getsize(fpath)//1024} KB)")

        fname = f'{layer_name}_b.h'
        fpath = os.path.join(WEIGHTS_DIR, fname)
        lines = [
            f'#pragma once',
            f'#include <cstdint>',
            f'',
            f'#define {macro_base}_B_SCALE {d["b_scale"]:.8f}f',
            f'#define {macro_base}_B_ZP    {d["b_zp"]}',
            f'',
            f'static const int8_t {layer_name}_b[{len(b)}] = ',
            array_to_c_initializer(b) + ';',
        ]
        with open(fpath, 'w') as f:
            f.write('\n'.join(lines))
        all_header_files.append(fname)

    # ── BatchNorm layer ───────────────────────────────────────────────────
    elif ltype == 'BatchNorm':
        # BatchNorm parameters stay as float32 for numerical stability
        # In hardware, BN can be fused with the preceding conv layer
        gamma  = d['gamma']
        beta   = d['beta']
        m_mean = d['mean']
        m_var  = d['var']
        eps    = 1e-3

        # Pre-compute fused BN scale and bias:
        #   scale_fused = gamma / sqrt(var + eps)
        #   bias_fused  = beta - gamma * mean / sqrt(var + eps)
        # This simplifies hardware: just multiply then add
        scale_fused = gamma / np.sqrt(m_var + eps)
        bias_fused  = beta - gamma * m_mean / np.sqrt(m_var + eps)

        fname = f'{layer_name}_bn.h'
        fpath = os.path.join(WEIGHTS_DIR, fname)

        def float_arr_to_c(arr, name):
            vals = ', '.join(f'{v:.8f}f' for v in arr.flatten())
            return f'static const float {name}[{len(arr)}] = {{{vals}}};'

        lines = [
            f'// Auto-generated — Layer: {layer_name}  Type: BatchNorm',
            f'// Pre-fused: scale = gamma/sqrt(var+eps), bias = beta - gamma*mean/sqrt(var+eps)',
            f'// Apply as: output = input * scale_fused + bias_fused',
            f'',
            f'#pragma once',
            f'',
            f'#define {macro_base}_CHANNELS {len(gamma)}',
            f'',
            float_arr_to_c(scale_fused, f'{layer_name}_scale'),
            float_arr_to_c(bias_fused,  f'{layer_name}_bias'),
        ]
        with open(fpath, 'w') as f:
            f.write('\n'.join(lines))
        all_header_files.append(fname)
        print(f"  Written: {fname}")

print(f"\n  Total header files written: {len(all_header_files)}")
print("  Step 5 complete.")


# This file contains all architecture constants that your C++ CNN code needs.
# Instead of hardcoding numbers in cnn.cpp, you #include model_config.h
# and use the named constants — makes the code cleaner and easier to modify.

print("\n" + "="*60)
print("STEP 6 — Writing model_config.h")
print("="*60)

config_path = os.path.join(WEIGHTS_DIR, 'model_config.h')

config_content = """// =================================================================
// model_config.h — Auto-generated by Phase 3 quantization script
// Architecture constants for Hybrid-Parallel 1D-CNN
// Use this in your Vitis HLS cnn.cpp project
// =================================================================

#pragma once

// ── Input ────────────────────────────────────────────────────────
#define INPUT_LENGTH     186      // timesteps per ECG beat
#define INPUT_CHANNELS   1        // single lead ECG
#define NUM_CLASSES      5        // N, S, V, F, Q

// ── Stage 1 (parallel pair, K=7, F=8) ───────────────────────────
#define S1_KERNEL        7
#define S1_FILTERS       8
#define S1_POOL          2        // MaxPool stride

// ── Stage 2 (parallel pair, K=7, F=16) ──────────────────────────
#define S2_KERNEL        7
#define S2_FILTERS       16
#define S2_POOL          2

// ── Stage 3 (sequential, K=5, F=32) ────────────────────────────
#define S3_KERNEL        5
#define S3_FILTERS       32
#define S3_POOL          2

// ── Stage 4 (sequential, K=5, F=64) ────────────────────────────
#define S4_KERNEL        5
#define S4_FILTERS       64
#define S4_POOL          2

// ── Fully Connected ─────────────────────────────────────────────
#define FC1_NEURONS      64
#define FC2_NEURONS      32
#define OUTPUT_NEURONS   5

// ── Quantization ────────────────────────────────────────────────
#define WEIGHT_BITS      8        // INT8 weights
#define ACCUM_BITS       32       // 32-bit accumulator to avoid overflow
#define USE_BATCH_NORM   1        // BatchNorm layers present (fused)

// ── FPGA-specific ────────────────────────────────────────────────
// These map to PYNQ-Z2 BRAM dimensions
// Use in HLS: #pragma HLS ARRAY_PARTITION variable=weights complete
#define MAX_FEATURE_MAP  186      // largest feature map dimension
#define MAX_FILTERS      64       // largest filter count
"""

with open(config_path, 'w') as f:
    f.write(config_content)

all_header_files.insert(0, 'model_config.h')
print(f"  Written: model_config.h")
print(config_content)


# This is the single file you #include in your cnn.cpp.
# It pulls in model_config.h and all individual weight headers.

print("\n" + "="*60)
print("STEP 7 — Writing master weights.h")
print("="*60)

master_path = os.path.join(WEIGHTS_DIR, 'weights.h')

# Sort headers sensibly
weight_headers = sorted([f for f in all_header_files
                         if f not in ('model_config.h', 'weights.h')])

master_lines = [
    '// =================================================================',
    '// weights.h — Master include file (auto-generated)',
    '// Add this single #include to your cnn.cpp in Vitis HLS',
    '// =================================================================',
    '',
    '#pragma once',
    '',
    '#include "model_config.h"',
    '',
    '// Individual layer weights',
]
for h in weight_headers:
    master_lines.append(f'#include "{h}"')

with open(master_path, 'w') as f:
    f.write('\n'.join(master_lines))

all_header_files.append('weights.h')
print(f"  Written: weights.h")
print(f"  Includes {len(weight_headers)} layer files + model_config.h")

# Print the contents
print("\n  File contents:")
for line in master_lines:
    print(f"    {line}")


# Summary of all scale and zero_point values in one place.
# Useful during C++ debugging to check quantization values.

print("\n" + "="*60)
print("STEP 8 — Writing quantization_params.h")
print("="*60)

qp_path = os.path.join(WEIGHTS_DIR, 'quantization_params.h')
qp_lines = [
    '// quantization_params.h — Summary of all INT8 quantization parameters',
    '// scale and zero_point for each Conv/Dense layer',
    '#pragma once',
    '',
    '// Format: {layer_name}_W_SCALE, {layer_name}_W_ZP',
    '// Dequantization: float = (int8 - zero_point) * scale',
    '',
]

for name, d in layer_data.items():
    if d['type'] in ('Conv1D', 'Dense'):
        macro = name.upper().replace('-','_')
        qp_lines.append(f'// {name}')
        qp_lines.append(f'//   W: scale={d["w_scale"]:.8f}  zp={d["w_zp"]}')
        if d['b_int8'] is not None:
            qp_lines.append(f'//   B: scale={d["b_scale"]:.8f}  zp={d["b_zp"]}')
        qp_lines.append('')

with open(qp_path, 'w') as f:
    f.write('\n'.join(qp_lines))

all_header_files.append('quantization_params.h')
print(f"  Written: quantization_params.h")
print("  Step 8 complete.")



print("\n" + "="*60)
print("STEP 9 — File summary")
print("="*60)

total_size = 0
print(f"\n  Files in {WEIGHTS_DIR}:")
print(f"  {'File':<35} {'Size':>10}")
print("  " + "-"*48)

for fname in sorted(os.listdir(WEIGHTS_DIR)):
    fpath = os.path.join(WEIGHTS_DIR, fname)
    size  = os.path.getsize(fpath)
    total_size += size
    print(f"  {fname:<35} {size/1024:>8.1f} KB")

print(f"  {'─'*48}")
print(f"  {'TOTAL':<35} {total_size/1024:>8.1f} KB")


# Creates one ZIP that you can download and directly use in Vitis HLS.
# Just unzip into your Vitis HLS project's source folder.

print("\n" + "="*60)
print("STEP 10 — Creating ZIP package for Vitis HLS")
print("="*60)

zip_path = os.path.join(DRIVE_ROOT, 'phase3_vitis_hls_weights.zip')

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Add all header files
    for fname in os.listdir(WEIGHTS_DIR):
        fpath = os.path.join(WEIGHTS_DIR, fname)
        zf.write(fpath, f'weights/{fname}')

    # Add TFLite model
    zf.write(tflite_path, 'model_int8.tflite')

    # Add a README
    readme = f"""
README — Phase 3 Output Package
================================

Contents:
  weights/model_config.h        — architecture constants
  weights/weights.h             — master include (add this to cnn.cpp)
  weights/*.h                   — individual layer weights (INT8)
  model_int8.tflite             — INT8 TFLite model (for Python inference)

How to use in Vitis HLS:
  1. Copy the weights/ folder into your Vitis HLS project src/ directory
  2. In cnn.cpp, add at the top:
       #include "model_config.h"
       #include "weights.h"
  3. All weight arrays and constants are now available

Quantization:
  - Weights: asymmetric INT8 (int8_t)
  - BatchNorm: pre-fused float32 (scale_fused, bias_fused)
  - Biases: INT8 where applicable

Model performance:
  Float32 accuracy : {float_acc*100:.2f}%
  INT8    accuracy : {int8_acc*100:.2f}%
  Accuracy drop    : {(float_acc-int8_acc)*100:.2f}%

Generated by Phase 3 quantization script.
"""
    zf.writestr('README.txt', readme)

zip_size = os.path.getsize(zip_path) / 1024
print(f"  ZIP created : phase3_vitis_hls_weights.zip")
print(f"  ZIP size    : {zip_size:.1f} KB")
print(f"  Saved to    : {zip_path}")



print("\n" + "="*60)
print("STEP 11 — Visualising weight distributions")
print("="*60)

# Pick first conv layer and output dense layer for comparison
conv_name  = [n for n, d in layer_data.items() if d['type']=='Conv1D'][0]
dense_name = [n for n, d in layer_data.items() if d['type']=='Dense'][-1]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Float32 vs INT8 Weight Distributions', fontsize=13, fontweight='bold')

for row, lname in enumerate([conv_name, dense_name]):
    d      = layer_data[lname]
    w_f    = d['w_float'].flatten()
    w_i    = d['w_int8'].flatten()
    w_deq  = dequantize(d['w_int8'], d['w_scale'], d['w_zp']).flatten()

    axes[row, 0].hist(w_f, bins=60, color='#4472C4', alpha=0.7, edgecolor='none')
    axes[row, 0].set_title(f'{lname} — Float32 weights', fontsize=10)
    axes[row, 0].set_xlabel('Weight value')
    axes[row, 0].set_ylabel('Count')

    axes[row, 1].hist(w_i, bins=60, color='#ED7D31', alpha=0.7, edgecolor='none')
    axes[row, 1].set_title(f'{lname} — INT8 weights', fontsize=10)
    axes[row, 1].set_xlabel('INT8 value (-128 to 127)')
    axes[row, 1].set_ylabel('Count')

plt.tight_layout()
fig_path = os.path.join(FIG_DIR, '01_weight_distributions.png')
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f"  Saved → 01_weight_distributions.png")



print("\n" + "="*60)
print("PHASE 3 COMPLETE — SUMMARY")
print("="*60)

print(f"""
  Quantization method  : Post-Training INT8 (asymmetric)
  Float32 accuracy     : {float_acc*100:.2f}%
  INT8    accuracy     : {int8_acc*100:.2f}%
  Accuracy drop        : {(float_acc-int8_acc)*100:.2f}%
  Size reduction       : {float_size/tflite_size:.1f}x smaller

  Files saved to Google Drive:
    phase3_weights/         ← C++ header files for Vitis HLS
      model_config.h        ← architecture constants
      weights.h             ← master include
      *_w.h, *_b.h, *_bn.h ← per-layer weight arrays
      quantization_params.h ← scale/zero_point reference

    phase3_tflite/
      model_int8.tflite     ← INT8 TFLite model

    phase3_vitis_hls_weights.zip  ← DOWNLOAD THIS for Phase 4

  ─────────────────────────────────────────────────────────
  NEXT STEP — Phase 4: Vitis HLS (on your PC)

  1. Download phase3_vitis_hls_weights.zip from Drive
  2. Install Xilinx Vitis HLS 2022.x on your PC
  3. Create new HLS project — target: xc7z020-1clg400c
  4. Unzip weights into your project src/ folder
  5. Write cnn.cpp using the weight headers
     (Phase 4 script provided next)
  ─────────────────────────────────────────────────────────
""")

Mounted at /content/drive
Drive mounted.
Model dir   : /content/drive/MyDrive/ECG_FPGA_Project/models
Weights dir : /content/drive/MyDrive/ECG_FPGA_Project/phase3_weights
TFLite dir  : /content/drive/MyDrive/ECG_FPGA_Project/phase3_tflite
TensorFlow version : 2.19.0

STEP 1 — Loading model and test data


  Model loaded : /content/drive/MyDrive/ECG_FPGA_Project/models/best_model.h5
  Parameters   : 25,693
  X_test       : (21892, 186, 1)   dtype: float32
  y_test       : (21892,)

  Evaluating float32 model on test set ...
  Float32 test accuracy : 94.52%
  Step 1 complete.

STEP 2 — TFLite INT8 Post-Training Quantization
  Converting model to INT8 TFLite ...
  Using 200 calibration samples ...
Saved artifact at '/tmp/tmpy09ktu0s'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 186, 1), dtype=tf.float32, name='ecg_input')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  140240843307152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140240843306192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140240843308880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140240843308688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140240843306960: TensorSpec(shape=(), dtype=t

In [ ]:
"""
=============================================================================
PHASE 4 — GENERATE TESTBENCH DATA (Run on Google Colab, CPU runtime)
=============================================================================
Run this BEFORE going to your PC.
It generates the two data files needed by testbench.cpp in Vitis HLS.

Output files (download these to PC):
  testbench_data/in.dat       → 100 ECG beats (186 floats each)
  testbench_data/expected.dat → 100 class labels (0-4)

Place both files inside your Vitis HLS project at:
  <project>/testbench_data/in.dat
  <project>/testbench_data/expected.dat
=============================================================================
"""


from google.colab import drive, files
drive.mount('/content/drive')

import numpy as np
import os

DRIVE_ROOT = '/content/drive/MyDrive/ECG_FPGA_Project'

# Load test data (saved by Phase 1)
X_test = np.load(os.path.join(DRIVE_ROOT, 'X_test.npy'))
y_test = np.load(os.path.join(DRIVE_ROOT, 'y_test.npy'))

print(f"X_test shape : {X_test.shape}")
print(f"y_test shape : {y_test.shape}")

# Pick 20 samples per class (100 total) for balanced testbench
N_PER_CLASS = 20
selected_idx = []

for cls in range(5):
    cls_idx = np.where(y_test == cls)[0]
    chosen  = cls_idx[:N_PER_CLASS]    # first N_PER_CLASS of each class
    selected_idx.extend(chosen.tolist())

selected_idx = sorted(selected_idx)   # keep original order
N = len(selected_idx)

print(f"\nSelected {N} samples ({N_PER_CLASS} per class)")
from collections import Counter
print(f"Class distribution: {Counter(y_test[selected_idx])}")

# Write in.dat
os.makedirs('testbench_data', exist_ok=True)

with open('testbench_data/in.dat', 'w') as f:
    for i in selected_idx:
        line = ' '.join(f'{v:.6f}' for v in X_test[i, :, 0])
        f.write(line + '\n')

# Write expected.dat
with open('testbench_data/expected.dat', 'w') as f:
    for i in selected_idx:
        f.write(str(int(y_test[i])) + '\n')

in_size  = os.path.getsize('testbench_data/in.dat')       / 1024
exp_size = os.path.getsize('testbench_data/expected.dat') / 1024

print(f"\nFiles created:")
print(f"  in.dat       : {in_size:.1f} KB  ({N} samples × 186 values)")
print(f"  expected.dat : {exp_size:.1f} KB  ({N} labels)")

# Download to PC
print("\nDownloading files to your PC ...")
files.download('testbench_data/in.dat')
files.download('testbench_data/expected.dat')
print("Done! Place both files in your Vitis HLS project at:")
print("  <project_root>/testbench_data/in.dat")
print("  <project_root>/testbench_data/expected.dat")


Mounted at /content/drive
X_test shape : (21892, 186, 1)
y_test shape : (21892,)

Selected 100 samples (20 per class)
Class distribution: Counter({np.int64(0): 20, np.int64(1): 20, np.int64(2): 20, np.int64(3): 20, np.int64(4): 20})

Files created:
  in.dat       : 173.7 KB  (100 samples × 186 values)
  expected.dat : 0.2 KB  (100 labels)



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Place both files in your Vitis HLS project at:
  <project_root>/testbench_data/in.dat
  <project_root>/testbench_data/expected.dat


In [ ]:
"""
=============================================================================
PHASE 6 — GENERATE TEST DATA FOR PYNQ BOARD
Run this on Google Colab BEFORE going to the board.
Downloads two files: X_pynq.npy and y_pynq.npy
Upload both to /home/xilinx/jupyter_notebooks/ on the PYNQ-Z2 board.
=============================================================================
"""

# Run on Colab (CPU runtime is fine)

from google.colab import drive, files
drive.mount('/content/drive')

import numpy as np
import os

DRIVE_ROOT = '/content/drive/MyDrive/ECG_FPGA_Project'

# Load test data
X_test = np.load(os.path.join(DRIVE_ROOT, 'X_test.npy'))
y_test = np.load(os.path.join(DRIVE_ROOT, 'y_test.npy'))

print(f"X_test : {X_test.shape}")
print(f"y_test : {y_test.shape}")

# Select 100 samples per class (500 total) for balanced evaluation
N_PER_CLASS = 100
idx = []
for c in range(5):
    ci = np.where(y_test == c)[0][:N_PER_CLASS]
    idx.extend(ci.tolist())

X_sub = X_test[idx]          # shape (500, 186, 1)
y_sub = y_test[idx]          # shape (500,)

# Flatten to (500, 186) — PYNQ script expects 1D beats
X_sub = X_sub[:, :, 0]      # shape (500, 186)

from collections import Counter
print(f"\nSelected {len(idx)} samples")
print(f"Class distribution: {Counter(y_sub)}")

np.save('X_pynq.npy', X_sub.astype(np.float32))
np.save('y_pynq.npy', y_sub.astype(np.int32))

print(f"\nSaved: X_pynq.npy  shape={X_sub.shape}  dtype=float32")
print(f"Saved: y_pynq.npy  shape={y_sub.shape}  dtype=int32")

print("\nDownloading...")
files.download('X_pynq.npy')
files.download('y_pynq.npy')

print("\nUpload both files to the PYNQ-Z2 board at:")
print("  http://192.168.2.99  (Jupyter file browser)")
print("  Upload to: /home/xilinx/jupyter_notebooks/")


Mounted at /content/drive
X_test : (21892, 186, 1)
y_test : (21892,)

Selected 500 samples
Class distribution: Counter({np.int64(0): 100, np.int64(1): 100, np.int64(2): 100, np.int64(3): 100, np.int64(4): 100})

Saved: X_pynq.npy  shape=(500, 186)  dtype=float32
Saved: y_pynq.npy  shape=(500,)  dtype=int32

Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Upload both files to the PYNQ-Z2 board at:
  http://192.168.2.99  (Jupyter file browser)
  Upload to: /home/xilinx/jupyter_notebooks/
